# Batch Object Gaussian Splat Pipeline

This notebook runs the **V12 all-instance bounded-volumetric object batch**. It discovers every
audited persistent instance belonging to the configured compact-object taxonomy and trains the
accepted V9 `ellipsoid_constrained` method for each object in the full-room coordinate system.

Outputs are keyed by persistent instance, so multiple chairs, pillows, bags, or other same-class
objects remain independently selectable. Planar, hanging, deformable, and fixture labels are
reported but excluded because they require structural or class-specific constraints rather than
the compact ellipsoid prior. Existing V9, V10, and V11 Drive outputs are never overwritten.

The dependency preflight pins a tested NumPy ABI before importing NumPy, and geometry sampling is
deterministic. A failed object is recorded without hiding the successful objects, and reruns reuse
completed per-instance dataset, training, and export stages when their fingerprints still match.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime
import copy
import csv
import hashlib
import json
import logging
import math
import os
import re
import shlex
from collections import deque
import shutil
import subprocess
import sys

BATCH_PIPELINE_VERSION = '2026-08-23-object-splat-all-volumetric-instances-v12'
RUNTIME_COMPATIBILITY_PATCH = '2026-08-23-python312-colab-runtime-gate-v2'
RESUME_EXPORT_PATCH = '2026-08-23-resumable-local-checkpoint-export-v1'
INTERRUPTION_RECOVERY_PATCH = '2026-08-23-incremental-manifest-recovery-v1'
CLEAN_LOCAL_OBJECT_CACHE_AFTER_SUCCESS = True

PROJECT_ROOT = Path('/content/drive/MyDrive/room-model-project')
SCAN_ID = 'scan_001'
SCAN_DIR = PROJECT_ROOT / 'scans' / SCAN_ID
SOURCE_NS_DATA_DIR = SCAN_DIR / 'nerfstudio-data'
DETECTION_DIR = SCAN_DIR / 'semantic_recon' / 'detections'
ROOM_SEMANTIC_DIR = SCAN_DIR / 'semantic_recon' / 'projection'
ROOM_LABELED_SPLAT_PATH = ROOM_SEMANTIC_DIR / 'splat_labeled.ply'
ROOM_SEMANTIC_LABELS_PATH = ROOM_SEMANTIC_DIR / 'splat_semantic_labels.json'
MASK_DIRS = [
    SCAN_DIR / 'semantic_recon' / 'segmentation' / 'masks',
    SCAN_DIR / 'semantic_recon' / 'layout' / 'masks',
]

# The v7 batch trusts only accepted records published by tracked segmentation v2.
# Filename fallback is disabled because old/orphaned PNG files can survive a rerun.
REQUIRE_TRACKED_SEGMENTATION_V2 = True
ALLOW_FILENAME_MASK_FALLBACK = False

# V12 discovers every audited instance from an explicit compact-volume taxonomy.
VOLUMETRIC_BATCH_MODE = True
AUTO_DISCOVER_VOLUMETRIC_LABELS = True
TARGET_LABELS = []  # Optional class filter; empty means every discovered allowlisted class.
VOLUMETRIC_LABEL_ALLOWLIST = {
    'bed', 'chair', 'table', 'tv', 'wardrobe', 'cabinet', 'drawer', 'shelf',
    'desk', 'sofa', 'lamp', 'pillow', 'bag', 'shoe', 'bottle', 'book', 'plant',
    'miscellaneous',
}
VOLUMETRIC_LABEL_EXCLUSIONS = {
    'wall': 'structural_plane',
    'floor': 'structural_plane',
    'ceiling': 'structural_plane',
    'door': 'architectural_plane',
    'window': 'architectural_plane',
    'curtain': 'hanging_deformable_surface',
    'rug': 'floor_surface',
    'blanket': 'deformable_surface',
    'mirror': 'thin_reflective_surface',
    'picture_frame': 'thin_wall_mounted_surface',
    'fan': 'thin_or_articulated_fixture',
    'light': 'ceiling_or_wall_fixture',
}
STRUCTURAL_OR_THIN_LABELS = set(VOLUMETRIC_LABEL_EXCLUSIONS)
# Optional per-class instance filter. Values may be one instance ID or a list of IDs.
# Empty means every qualifying persistent instance of every selected class.
TARGET_INSTANCE_IDS = {}
INSTANCE_SELECTION_POLICY = 'all_qualifying_persistent_instances'
RESOLVED_INSTANCE_IDS = {}
TARGET_OBJECT_LABELS = {}
TARGET_OBJECTS = []
DISCOVERED_VOLUMETRIC_LABELS = []

# The three arms share identical cameras, RGB images, masks, and sparse initialization.
PILOT_VARIANTS = {
    'masked_baseline': {
        'method': 'splatfacto',
        'description': 'Stock Nerfstudio mask-as-ignore baseline.',
    },
    'alpha_constrained': {
        'method': 'object-alpha-splatfacto',
        'description': 'Explicit opacity target and mask-gated densification.',
    },
    'ellipsoid_constrained': {
        'method': 'object-ellipsoid-splatfacto',
        'description': 'Alpha constraints plus oriented 2-sigma ellipsoid containment.',
    },
}
PILOT_VARIANTS_TO_RUN = ['ellipsoid_constrained']  # Use all three only for an explicit ablation.
PILOT_RECOMMENDED_VARIANT = 'ellipsoid_constrained'

# Batch controls.
# Keep this stable to resume a batch across Colab disconnects. Change it intentionally for a new experiment.
BATCH_ID = 'all_volumetric_instances_ellipsoid_batch_v12'
MAX_OBJECTS_TO_RUN = None  # Example: 2 for a short smoke test.
STOP_ON_LABEL_FAILURE = False
SKIP_LABELS_WITH_TOO_FEW_MASK_FRAMES = True
MIN_POSITIVE_MASK_FRAMES = 5

# Coordinate safety is non-negotiable. The default retains the complete registered camera set,
# which makes Nerfstudio compute the same orientation, center, and scale as the room model.
# A future reduced-view mode must first reproduce the room dataparser transform exactly.
MAX_FRAMES = None
KEEP_ALL_REGISTERED_FRAMES_FOR_SHARED_COORDINATES = True
EMPTY_MASK_FOR_MISSING_TARGET_FRAMES = True
SHARED_COORDINATE_POLICY = 'full_room_camera_set_identical_matrices_and_metadata'
VERIFY_COORDINATE_INVARIANTS = True

# Runtime profile. Images are staged from Drive once and shared by every label through local
# symlinks. Per-label datasets write masks only; they never duplicate RGB image pyramids.
LOCAL_CACHE_ROOT = Path('/content/room-model-object-splat-cache') / SCAN_ID
LOCAL_SHARED_DATA_DIR = LOCAL_CACHE_ROOT / 'shared-nerfstudio-data'
REUSE_LOCAL_SHARED_CACHE = True
REUSE_COMPLETED_DATASET_STAGE = True
PERSIST_PER_LABEL_DATASETS_TO_DRIVE = False
TRAINING_PROFILE = 'quality'  # 'smoke', 'quality', or 'final'
TRAINING_PROFILE_ITERATIONS = {'smoke': 1000, 'quality': 5000, 'final': 10000}

# Mask controls. Stricter masks reduce the background shell around each object splat.
# Surfaces like floor/ceiling should usually not erode; object labels can tolerate a light erosion.
MIN_MASK_AREA_FRACTION = 0.0001
MASK_DILATION_PIXELS = 0
MASK_EROSION_PIXELS = 0
MASK_DILATION_PIXELS_BY_LABEL = {
    'floor': 0,
    'wall': 0,
    'ceiling': 0,
    'door': 0,
}
MASK_EROSION_PIXELS_BY_LABEL = {
    'floor': 0,
    'wall': 0,
    'ceiling': 0,
    'door': 0,
}
WRITE_MASKED_RGB_PREVIEWS = False
PREVIEW_FRAME_LIMIT = 8

# Sparse initialization. Requiring support in more than one masked view reduces background seeding.
CREATE_OBJECT_SPARSE_INIT = True
SPARSE_POINT_MIN_MASK_HITS = 2
SPARSE_POINT_CHUNK_SIZE = 200_000
SPARSE_POINT_MAX_OUTPUT_POINTS = 250_000
REMOVE_FULL_SCENE_PLY_INIT_IF_NO_OBJECT_INIT = True
CAMERA_CONVENTION = 'nerfstudio_opengl'

# Training/export controls. The first batch was a proof run; these defaults target cleaner final object splats.
RUN_TRAINING = True
RUN_EXPORT = True
SKIP_EXISTING_SUCCESSFUL_EXPORTS = True
# Reuse the previous batch's raw exports to test the new cleaner without retraining.
# Set to None when masks/training have changed and fresh object models are required.
REUSE_RAW_EXPORTS_FROM_BATCH_ID = None
TRAIN_ITERATIONS = TRAINING_PROFILE_ITERATIONS[TRAINING_PROFILE]
TRAIN_ITERATIONS_BY_LABEL = {
    'floor': 12000,
    'wall': 12000,
    'ceiling': 12000,
}
STEPS_PER_SAVE = 5000
DOWNSCALE_FACTOR = 2
DOWNSCALE_FACTORS_TO_WRITE = sorted({1, int(DOWNSCALE_FACTOR)})


# Explicit accumulation and extent supervision for the custom methods.
PILOT_ALPHA_POSITIVE_LOSS_MULT = 0.20
PILOT_ALPHA_OUTSIDE_LOSS_MULT = 0.25
PILOT_ALPHA_DICE_LOSS_MULT = 0.10
PILOT_OUTSIDE_TOLERANCE_PIXELS = 2
PILOT_MAX_SIGMA_VOXELS = 1.25
PILOT_HULL_SUPPORT_LOSS_MULT = 0.05
PILOT_HULL_BBOX_LOSS_MULT = 0.10
PILOT_HULL_EXTENT_LOSS_MULT = 0.05
PILOT_HULL_PROJECTION_INTERVAL = 100


# V9 constrains the learned WXYZ-rotated Gaussian contour, not just its center. The finite
# object boundary is defined at two standard deviations and sampled on two radial shells.
PILOT_ELLIPSOID_SIGMA_MULTIPLIER = 2.0
PILOT_ELLIPSOID_FIBONACCI_DIRECTION_COUNT = 42  # Plus six principal-axis probes.
PILOT_ELLIPSOID_RADIAL_SHELL_COUNT = 2
PILOT_ELLIPSOID_SUPPORT_LOSS_MULT = 0.10
PILOT_ELLIPSOID_BBOX_LOSS_MULT = 0.10
PILOT_ELLIPSOID_MIN_OCCUPANCY = 0.50
PILOT_ELLIPSOID_LOSS_SAMPLE_COUNT = 1024
PILOT_ELLIPSOID_LOSS_EVERY = 2
PILOT_ELLIPSOID_HARD_CHUNK_SIZE = 16384
PILOT_ELLIPSOID_SHRINK_SEARCH_STEPS = 7
PILOT_ELLIPSOID_SNAP_THRESHOLD = 0.10


# Conservative multi-view support-grid construction. Positive visible masks define support;
# blank masks are never interpreted as proof that an occluded object does not exist.
HULL_POSITIVE_FRAME_LIMIT = 48
HULL_PROJECTION_CHUNK_SIZE = 150_000
HULL_VOXEL_SPACING_MULTIPLIER = 1.5
HULL_MAX_VOXELS = 350_000
HULL_MAX_AXIS_CELLS = 112
HULL_PADDING_FRACTION = 0.12
HULL_PADDING_SPACING_MULTIPLIER = 4.0
HULL_MASK_DILATION_PIXELS = 3
HULL_MIN_SUPPORTING_VIEWS = 2
HULL_MIN_SUPPORT_RATIO = 0.35
HULL_SEED_PROTECTION_RADIUS_VOXELS = 1
HULL_MIN_OCCUPIED_CELLS = 64

# Room-reference cleanup. The labeled full-room splat defines occupied geometry and conservative
# semantic conflicts. Unknown or low-confidence room labels never justify deletion.
# First evaluate what the improved masks train without a post-export confound.
RUN_POST_EXPORT_CLEANUP = False
PREFER_CLEANED_SPLAT_IN_MANIFEST = False
POST_EXPORT_MIN_ALPHA = 0.02
POST_EXPORT_MIN_GAUSSIAN_COUNT = 500
# A selected candidate must retain this fraction of the alpha-only population. This prevents a
# tiny remnant from being called valid merely because it contains 500 Gaussians.
CLEANUP_MIN_COMPLETENESS_VS_ALPHA = 0.70

# Full-room semantic reference. These fields are written by pipeline.ipynb into splat_labeled.ply.
ROOM_REFERENCE_REQUIRED = False
ROOM_REFERENCE_MIN_CONFIDENCE = 0.80
ROOM_REFERENCE_MIN_SUPPORT_VIEWS = 4
ROOM_REFERENCE_MIN_COMPETING_NEIGHBORS = 4
ROOM_REFERENCE_MIN_ROOM_NEIGHBORS = 1
ROOM_REFERENCE_TARGET_PROTECTION_NEIGHBORS = 1
ROOM_REFERENCE_MIN_TARGET_POINTS = 20
ROOM_REFERENCE_RADIUS_SPACING_MULTIPLIER = 2.5
ROOM_REFERENCE_RADIUS_SIGMA_MULTIPLIER = 2.0
ROOM_REFERENCE_MIN_RADIUS = 1e-4
ROOM_REFERENCE_MAX_RADIUS_PERCENTILE = 95.0
RUN_LEGACY_IMAGE_CLEANUP_DIAGNOSTICS = False

# Multiview Gaussian-footprint support. This protects useful large/elongated Gaussians even when
# their centers are not close to sparse points.
CLEANUP_POSITIVE_FRAME_LIMIT = 32
CLEANUP_NEGATIVE_FRAME_LIMIT = 8
CLEANUP_PROJECTION_CHUNK_SIZE = 250_000
CLEANUP_MIN_SUPPORTING_VIEWS = 3
CLEANUP_MIN_MASK_SUPPORT_RATIO = 0.50
CLEANUP_MIN_FOOTPRINT_COVERAGE = 0.20
CLEANUP_MAX_CONTRADICTION_RATIO = 0.55
# Deletion requires strong background evidence. Failure to prove foreground membership is uncertainty,
# not evidence of background.
CLEANUP_BACKGROUND_MIN_VISIBLE_VIEWS = 4
CLEANUP_BACKGROUND_MAX_SUPPORTING_VIEWS = 0
CLEANUP_BACKGROUND_MIN_CONTRADICTION_RATIO = 0.80
CLEANUP_NEGATIVE_VIEW_WEIGHT = 0.35
CLEANUP_ZBUFFER_ABS_TOLERANCE = 0.08
CLEANUP_ZBUFFER_SIGMA_MULTIPLIER = 2.0
CLEANUP_MAX_FOOTPRINT_RADIUS_PX = 64.0

# Geometry support uses local sparse spacing plus Gaussian extent, rather than center distance alone.
CLEANUP_SPARSE_DENSITY_NEIGHBORS = 4
CLEANUP_SPARSE_DISTANCE_SPACING_MULTIPLIER = 3.0
CLEANUP_GAUSSIAN_EXTENT_MULTIPLIER = 3.0
CLEANUP_MAX_SIGMA_PERCENTILE = 97.0

# Weak disconnected fragments are removed after visual/geometric evidence filtering.
CLEANUP_COMPONENT_MIN_GAUSSIANS = 300
CLEANUP_COMPONENT_MIN_FRACTION = 0.001
CLEANUP_COMPONENT_MIN_STRONG_VISUAL_GAUSSIANS = 30
CLEANUP_VOXEL_SPACING_MULTIPLIER = 2.0

# Floor and ceiling are broad surfaces and use a robust plane constraint instead of component pruning.
STRUCTURAL_PLANE_LABELS = {'floor', 'ceiling'}
CLEANUP_PLANE_RANSAC_ITERATIONS = 256
CLEANUP_PLANE_SPACING_MULTIPLIER = 2.5

BATCH_ROOT = SCAN_DIR / 'object_splat_batches' / BATCH_ID
REPORT_DIR = SCAN_DIR / 'reports' / 'object_splat_batch' / BATCH_ID
OBJECT_SPLATS_DIR = SCAN_DIR / 'object_splats'
BATCH_MANIFEST_PATH = OBJECT_SPLATS_DIR / 'manifests' / f'{BATCH_ID}_object_splat_manifest.json'
LATEST_MANIFEST_PATH = OBJECT_SPLATS_DIR / 'object_splat_manifest.json'
FATAL_DIR = SCAN_DIR / 'reports' / 'fatal_errors'

for directory in [BATCH_ROOT, REPORT_DIR, OBJECT_SPLATS_DIR / 'manifests', FATAL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

LOG_PATH = REPORT_DIR / 'object_splat_batch.log'
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s',
    handlers=[logging.FileHandler(LOG_PATH, mode='w'), logging.StreamHandler()],
)
logger = logging.getLogger('object_splat_batch')

def fail(message):
    FATAL_DIR.mkdir(parents=True, exist_ok=True)
    with (FATAL_DIR / 'fatal_error.log').open('a') as f:
        f.write(f'[{datetime.now().isoformat(timespec="seconds")}] object_splat_batch: {message}\n')
    raise RuntimeError(message)

print('Batch pipeline version:', BATCH_PIPELINE_VERSION)
print('Batch id:', BATCH_ID)
print('Explicit target-label filter:', TARGET_LABELS or 'all allowlisted labels')
print('Containment variants:', PILOT_VARIANTS_TO_RUN)
print('Recommended variant:', PILOT_RECOMMENDED_VARIANT)
print('Batch root:', BATCH_ROOT)
print('Reports:', REPORT_DIR)
print('Latest manifest path:', LATEST_MANIFEST_PATH)
print('Room labeled splat:', ROOM_LABELED_SPLAT_PATH)
print('Room semantic labels:', ROOM_SEMANTIC_LABELS_PATH)
print('Require tracked segmentation v2:', REQUIRE_TRACKED_SEGMENTATION_V2)
print('Allow filename mask fallback:', ALLOW_FILENAME_MASK_FALLBACK)
print('Post-export cleanup enabled:', RUN_POST_EXPORT_CLEANUP)
logger.info('Batch pipeline version: %s; batch id: %s', BATCH_PIPELINE_VERSION, BATCH_ID)

## 3. Install/Validate Dependencies

In [ ]:
import importlib.util
from importlib.metadata import PackageNotFoundError, version as package_version

NERFSTUDIO_TESTED_VERSION = '1.1.5'
NUMPY_TESTED_VERSION = '1.26.4'
SUPPORTED_PYTHON_VERSIONS = {(3, 11), (3, 12)}
RECOMMENDED_COLAB_RUNTIME_VERSION = '2026.07'

python_version = sys.version_info[:2]
if python_version not in SUPPORTED_PYTHON_VERSIONS:
    raise RuntimeError(
        f'Python {sys.version.split()[0]} is incompatible with this tested '
        f'Nerfstudio/NumPy stack. In Colab choose Runtime > Change runtime type, '
        f'set Runtime Version to {RECOMMENDED_COLAB_RUNTIME_VERSION} and Hardware '
        'accelerator to T4 GPU, reconnect, then run the notebook from the top. '
        'Do not try to repair the current Python 3.13 session.'
    )


def installed_version(package_name):
    try:
        return package_version(package_name)
    except PackageNotFoundError:
        return None


numpy_version_before = installed_version('numpy')
numpy_loaded_before_install = 'numpy' in sys.modules
nerfstudio_version_before = installed_version('nerfstudio')
needs_install = (
    numpy_version_before != NUMPY_TESTED_VERSION
    or nerfstudio_version_before != NERFSTUDIO_TESTED_VERSION
    or importlib.util.find_spec('plyfile') is None
    or shutil.which('ns-train') is None
)

if needs_install:
    print(
        'Installing the tested NumPy/Nerfstudio environment:',
        f'numpy=={NUMPY_TESTED_VERSION},',
        f'nerfstudio=={NERFSTUDIO_TESTED_VERSION}',
    )
    pip_install = subprocess.run(
        [
            sys.executable, '-m', 'pip', 'install', '-q', '--no-input',
            '--upgrade-strategy', 'only-if-needed',
            f'numpy=={NUMPY_TESTED_VERSION}',
            'plyfile',
            f'nerfstudio=={NERFSTUDIO_TESTED_VERSION}',
        ],
        capture_output=True,
        text=True,
    )
    if pip_install.returncode != 0:
        print('pip installation stdout (tail):')
        print(pip_install.stdout[-6000:])
        print('pip installation stderr (tail):')
        print(pip_install.stderr[-6000:])
        raise RuntimeError(
            'Could not install the tested object-splat environment. Confirm that '
            f'Colab Runtime Version is {RECOMMENDED_COLAB_RUNTIME_VERSION}, then '
            'restart the session and rerun from the top.'
        )

numpy_version_after = installed_version('numpy')
nerfstudio_version_after = installed_version('nerfstudio')
if numpy_version_after != NUMPY_TESTED_VERSION:
    raise RuntimeError(
        f'Expected numpy=={NUMPY_TESTED_VERSION}, found {numpy_version_after}. '
        'Use Runtime > Restart session, then rerun from the top.'
    )
if nerfstudio_version_after != NERFSTUDIO_TESTED_VERSION:
    raise RuntimeError(
        f'Expected nerfstudio=={NERFSTUDIO_TESTED_VERSION}, '
        f'found {nerfstudio_version_after}.'
    )

if (
    needs_install
    and numpy_loaded_before_install
    and numpy_version_before != numpy_version_after
):
    raise RuntimeError(
        'A compatible NumPy build was installed, but the old NumPy binary is already '
        'loaded in this Colab process. Use Runtime > Restart session, '
        'then rerun the notebook from the top. Do not continue this runtime.'
    )

numpy_probe = subprocess.run(
    [
        sys.executable,
        '-c',
        (
            'import numpy as np; '
            'rng = np.random.default_rng(42); '
            'assert rng.integers(0, 10) >= 0; '
            'print(np.__version__)'
        ),
    ],
    capture_output=True,
    text=True,
)
if numpy_probe.returncode != 0:
    print('NumPy ABI probe stderr:')
    print(numpy_probe.stderr[-4000:])
    subprocess.run(
        [
            sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
            '--no-cache-dir', '--no-deps', f'numpy=={NUMPY_TESTED_VERSION}',
        ],
        check=True,
    )
    raise RuntimeError(
        'The NumPy installation was internally inconsistent and has been repaired on '
        'disk. Use Runtime > Restart session, then rerun from the top.'
    )

import cv2
import numpy as np
from tqdm.auto import tqdm
from plyfile import PlyData, PlyElement

print('NumPy version:', np.__version__)
print('NumPy ABI probe:', numpy_probe.stdout.strip())
print('Nerfstudio version:', nerfstudio_version_after)
print('ns-train:', shutil.which('ns-train'))
print('ns-export:', shutil.which('ns-export'))
if shutil.which('ns-train') is None:
    raise RuntimeError('ns-train is unavailable after install attempt.')
if shutil.which('ns-export') is None:
    raise RuntimeError('ns-export is unavailable after install attempt.')

if shutil.which('nvidia-smi') is not None:
    subprocess.run(['nvidia-smi'], check=False)
else:
    print('WARNING: nvidia-smi was not found. Use Colab Runtime > Change runtime type > GPU before training.')

try:
    import torch
    print('torch.cuda.is_available():', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('CUDA device:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('WARNING: Could not import/check torch:', exc)


### 3A. Register Embedded Object-Constrained Splatfacto

This cell writes the maintained V9 plugin to temporary Colab storage and validates the alpha,
center-hull compatibility, and oriented-ellipsoid method registrations. It does not write source
code dependencies to Google Drive.


In [ ]:
OBJECT_CONSTRAINED_PLUGIN_SOURCE = r'''"""Nerfstudio methods for the object-containment Gaussian pilot.

This module is embedded verbatim in ``object_splat_batch_pipeline.ipynb``. The
notebook writes it to Colab's temporary filesystem and registers the custom methods
through ``NERFSTUDIO_METHOD_CONFIGS``; Google Drive does not need this file.
"""

from __future__ import annotations

import copy
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, Optional, Type

import numpy as np
import torch
import torch.nn.functional as F
from plyfile import PlyData

from nerfstudio.configs.method_configs import method_configs
from nerfstudio.models.splatfacto import SplatfactoModel, SplatfactoModelConfig
from nerfstudio.plugins.types import MethodSpecification


@dataclass
class ObjectConstrainedSplatfactoModelConfig(SplatfactoModelConfig):
    """Splatfacto plus explicit silhouette and optional 3D containment constraints."""

    _target: Type = field(default_factory=lambda: ObjectConstrainedSplatfactoModel)

    alpha_positive_loss_mult: float = 0.20
    """Penalty for failing to render opacity inside a visible object mask."""

    alpha_outside_loss_mult: float = 0.25
    """Penalty for rendering opacity outside the relaxed object silhouette."""

    alpha_dice_loss_mult: float = 0.10
    """Whole-silhouette Dice penalty used with the two directional alpha losses."""

    outside_tolerance_pixels: int = 2
    """Mask dilation allowed before outside opacity is penalized."""

    minimum_positive_mask_pixels: int = 16
    """Blank and tiny masks preserve coordinates but do not carve occluded geometry."""

    gate_densification_by_mask: bool = True
    """Prevent center gradients outside the current silhouette from causing splits/clones."""

    enable_hull_constraint: bool = False
    support_grid_path: Optional[Path] = None
    raw_seed_path: Optional[Path] = None

    hull_support_loss_mult: float = 0.05
    hull_bbox_loss_mult: float = 0.10
    hull_min_occupancy: float = 0.50
    hull_loss_sample_count: int = 8192
    hull_loss_every: int = 1

    hard_project_outside_hull: bool = False
    hull_projection_interval: int = 100
    max_sigma_voxels: float = 1.25
    extent_loss_mult: float = 0.05
    projection_log_interval: int = 500

    enable_ellipsoid_constraint: bool = False
    """Constrain a finite oriented Gaussian contour, not only its center."""

    ellipsoid_sigma_multiplier: float = 2.0
    """Finite contour radius in standard deviations; a Gaussian has infinite support."""

    ellipsoid_fibonacci_direction_count: int = 42
    """Fibonacci-sphere probes; six principal-axis probes are added automatically."""

    ellipsoid_radial_shell_count: int = 2
    """Sample the outer contour and interior shells to catch unsupported cavities."""

    ellipsoid_support_loss_mult: float = 0.10
    ellipsoid_bbox_loss_mult: float = 0.10
    ellipsoid_min_occupancy: float = 0.50
    ellipsoid_loss_sample_count: int = 1024
    ellipsoid_loss_every: int = 2

    hard_constrain_ellipsoid: bool = False
    ellipsoid_hard_chunk_size: int = 16384
    ellipsoid_shrink_search_steps: int = 7
    ellipsoid_snap_threshold: float = 0.10
    """Snap a center to its occupied voxel before an extreme (>90%) scale reduction."""


class ObjectConstrainedSplatfactoModel(SplatfactoModel):
    """Object-aware Splatfacto used by the controlled bed pilot.

    The model keeps stock masked RGB supervision, but adds a separate target for
    the rendered accumulation image. For the hull arm, a conservative visual
    support grid is loaded in raw scan coordinates and aligned to Nerfstudio's
    internal training coordinates from the corresponding sparse seed points.
    The V9 arm additionally constrains sampled points throughout each learned
    quaternion-rotated finite-sigma ellipsoid.
    """

    config: ObjectConstrainedSplatfactoModelConfig

    def populate_modules(self):
        super().populate_modules()
        self._last_center_support: Optional[torch.Tensor] = None
        self._hull_loaded = False
        self._last_projected_count = 0
        self._total_projected_count = 0
        self._last_ellipsoid_shrunk_count = 0
        self._total_ellipsoid_shrunk_count = 0
        self._last_ellipsoid_snapped_count = 0
        self._total_ellipsoid_snapped_count = 0
        self._last_boundary_inside_fraction = 1.0

        if self.config.enable_ellipsoid_constraint and not self.config.enable_hull_constraint:
            raise RuntimeError("Ellipsoid containment requires the 3D support hull.")
        if self.config.enable_ellipsoid_constraint:
            if float(self.config.ellipsoid_sigma_multiplier) <= 0:
                raise RuntimeError("ellipsoid_sigma_multiplier must be positive.")
            if int(self.config.ellipsoid_radial_shell_count) < 1:
                raise RuntimeError("ellipsoid_radial_shell_count must be at least one.")
            if int(self.config.ellipsoid_loss_sample_count) < 1:
                raise RuntimeError("ellipsoid_loss_sample_count must be at least one.")

        if self.config.enable_hull_constraint:
            self.register_buffer(
                "_ellipsoid_directions",
                self._build_ellipsoid_directions(
                    int(self.config.ellipsoid_fibonacci_direction_count)
                ),
            )
            self._load_hull_constraint()

    @staticmethod
    def _build_ellipsoid_directions(fibonacci_count: int) -> torch.Tensor:
        """Return deterministic unit directions covering the complete ellipsoid surface."""

        fibonacci_count = max(int(fibonacci_count), 8)
        index = torch.arange(fibonacci_count, dtype=torch.float32)
        z = 1.0 - 2.0 * (index + 0.5) / float(fibonacci_count)
        radius = torch.sqrt(torch.clamp(1.0 - z.square(), min=0.0))
        golden_angle = torch.tensor(np.pi * (3.0 - np.sqrt(5.0)), dtype=torch.float32)
        theta = index * golden_angle
        fibonacci = torch.stack(
            [radius * torch.cos(theta), radius * torch.sin(theta), z], dim=-1
        )
        axes = torch.tensor(
            [
                [1.0, 0.0, 0.0],
                [-1.0, 0.0, 0.0],
                [0.0, 1.0, 0.0],
                [0.0, -1.0, 0.0],
                [0.0, 0.0, 1.0],
                [0.0, 0.0, -1.0],
            ],
            dtype=torch.float32,
        )
        return torch.cat([axes, fibonacci], dim=0)

    @staticmethod
    def _quat_wxyz_to_rotation_matrix(quaternions: torch.Tensor) -> torch.Tensor:
        """Convert gsplat's WXYZ quaternions to local-to-model rotation matrices."""

        q = F.normalize(quaternions, dim=-1)
        w, x, y, z = q.unbind(dim=-1)
        return torch.stack(
            [
                1.0 - 2.0 * (y.square() + z.square()),
                2.0 * (x * y - w * z),
                2.0 * (x * z + w * y),
                2.0 * (x * y + w * z),
                1.0 - 2.0 * (x.square() + z.square()),
                2.0 * (y * z - w * x),
                2.0 * (x * z - w * y),
                2.0 * (y * z + w * x),
                1.0 - 2.0 * (x.square() + y.square()),
            ],
            dim=-1,
        ).reshape(-1, 3, 3)

    @staticmethod
    def _read_xyz(path: Path) -> np.ndarray:
        ply = PlyData.read(str(path))
        vertex = ply["vertex"].data
        names = set(vertex.dtype.names or [])
        if not {"x", "y", "z"}.issubset(names):
            raise RuntimeError(f"Hull seed PLY is missing x/y/z: {path}")
        return np.column_stack(
            [
                np.asarray(vertex["x"], dtype=np.float64),
                np.asarray(vertex["y"], dtype=np.float64),
                np.asarray(vertex["z"], dtype=np.float64),
            ]
        )

    @staticmethod
    def _fit_affine(source: np.ndarray, target: np.ndarray) -> np.ndarray:
        source_h = np.column_stack([source, np.ones(len(source), dtype=np.float64)])
        coefficients, _, _, _ = np.linalg.lstsq(source_h, target, rcond=None)
        return coefficients.astype(np.float32)

    @staticmethod
    def _apply_affine_numpy(points: np.ndarray, coefficients: np.ndarray) -> np.ndarray:
        points_h = np.column_stack([points, np.ones(len(points), dtype=np.float64)])
        return points_h @ coefficients

    def _load_hull_constraint(self) -> None:
        if self.config.support_grid_path is None or self.config.raw_seed_path is None:
            raise RuntimeError(
                "Hull-constrained training requires support_grid_path and raw_seed_path."
            )

        grid_path = Path(self.config.support_grid_path)
        raw_seed_path = Path(self.config.raw_seed_path)
        if not grid_path.exists():
            raise FileNotFoundError(f"Missing object support grid: {grid_path}")
        if not raw_seed_path.exists():
            raise FileNotFoundError(f"Missing raw object sparse seed PLY: {raw_seed_path}")

        raw_seed = self._read_xyz(raw_seed_path)
        model_seed = self.means.detach().cpu().numpy().astype(np.float64)
        if len(raw_seed) != len(model_seed):
            raise RuntimeError(
                "Raw/model seed count mismatch; cannot recover the dataparser transform: "
                f"{len(raw_seed)} != {len(model_seed)}"
            )
        if len(raw_seed) < 4:
            raise RuntimeError("At least four object sparse seed points are required for hull alignment.")

        raw_to_model = self._fit_affine(raw_seed, model_seed)
        model_to_raw = self._fit_affine(model_seed, raw_seed)
        fitted = self._apply_affine_numpy(raw_seed, raw_to_model)
        fit_error = np.linalg.norm(fitted - model_seed, axis=1)
        model_span = float(np.linalg.norm(np.ptp(model_seed, axis=0)))
        fit_tolerance = max(1e-5, model_span * 1e-4)
        if not np.isfinite(fit_error).all() or float(np.max(fit_error)) > fit_tolerance:
            raise RuntimeError(
                "Could not align the raw support hull to Nerfstudio coordinates. "
                f"max residual={float(np.max(fit_error)):.6g}, tolerance={fit_tolerance:.6g}"
            )

        with np.load(grid_path, allow_pickle=False) as payload:
            occupancy = np.asarray(payload["occupancy"], dtype=bool)
            nearest = np.asarray(payload["nearest_occupied_index"], dtype=np.int32)
            origin = np.asarray(payload["origin"], dtype=np.float32)
            axes = np.asarray(payload["axes"], dtype=np.float32)
            voxel_size = float(np.asarray(payload["voxel_size"]).reshape(-1)[0])

        if occupancy.ndim != 3 or not occupancy.any():
            raise RuntimeError(f"Object support grid has no occupied 3D cells: {grid_path}")
        if nearest.shape != occupancy.shape + (3,):
            raise RuntimeError(
                f"Invalid nearest-index shape in {grid_path}: {nearest.shape} versus {occupancy.shape}"
            )
        if origin.shape != (3,) or axes.shape != (3, 3) or voxel_size <= 0:
            raise RuntimeError(f"Invalid support-grid geometry metadata: {grid_path}")

        self.register_buffer("_raw_to_model", torch.from_numpy(raw_to_model))
        self.register_buffer("_model_to_raw", torch.from_numpy(model_to_raw))
        self.register_buffer("_hull_origin", torch.from_numpy(origin))
        self.register_buffer("_hull_axes", torch.from_numpy(axes))
        self.register_buffer("_hull_occupancy", torch.from_numpy(occupancy))
        self.register_buffer("_hull_nearest", torch.from_numpy(nearest.astype(np.int64)))
        self.register_buffer(
            "_hull_shape",
            torch.tensor(occupancy.shape, dtype=torch.float32),
        )
        self.register_buffer("_hull_voxel_size", torch.tensor(voxel_size, dtype=torch.float32))

        raw_probe = np.vstack(
            [origin, *[origin + axes[:, axis] * voxel_size for axis in range(3)]]
        )
        model_probe = self._apply_affine_numpy(raw_probe, raw_to_model)
        model_voxel_sizes = np.linalg.norm(model_probe[1:] - model_probe[0], axis=1)
        model_voxel_size = float(np.median(model_voxel_sizes))
        if not np.isfinite(model_voxel_size) or model_voxel_size <= 0:
            raise RuntimeError("Could not derive the support-grid voxel size in model coordinates.")
        self.register_buffer(
            "_model_voxel_size",
            torch.tensor(model_voxel_size, dtype=torch.float32),
        )

        self._hull_loaded = True
        print(
            "[object-hull] loaded",
            tuple(int(value) for value in occupancy.shape),
            "occupied=",
            int(occupancy.sum()),
            "raw_voxel=",
            voxel_size,
            "model_voxel=",
            model_voxel_size,
            "alignment_max_error=",
            float(np.max(fit_error)),
        )

    @staticmethod
    def _apply_affine_torch(points: torch.Tensor, coefficients: torch.Tensor) -> torch.Tensor:
        ones = torch.ones((len(points), 1), dtype=points.dtype, device=points.device)
        return torch.cat([points, ones], dim=-1) @ coefficients.to(points)

    def _model_points_to_raw(self, points: torch.Tensor) -> torch.Tensor:
        return self._apply_affine_torch(points, self._model_to_raw)

    def _raw_points_to_model(self, points: torch.Tensor) -> torch.Tensor:
        return self._apply_affine_torch(points, self._raw_to_model)

    def _raw_points_to_grid(self, raw_points: torch.Tensor) -> torch.Tensor:
        local = (raw_points - self._hull_origin.to(raw_points)) @ self._hull_axes.to(raw_points)
        return local / self._hull_voxel_size.to(raw_points)

    def _lookup_hull(self, grid_points: torch.Tensor):
        rounded = torch.round(grid_points).long()
        shape = self._hull_shape.long()
        inside_bounds = ((rounded >= 0) & (rounded < shape)).all(dim=-1)
        clamped = torch.minimum(torch.maximum(rounded, torch.zeros_like(rounded)), shape - 1)
        occupied = self._hull_occupancy[
            clamped[:, 0], clamped[:, 1], clamped[:, 2]
        ]
        return inside_bounds & occupied, clamped

    def _sample_hull_occupancy(self, grid_points: torch.Tensor) -> torch.Tensor:
        denominator = torch.clamp(self._hull_shape.to(grid_points) - 1.0, min=1.0)
        normalized = 2.0 * grid_points / denominator - 1.0
        sample_grid = normalized.view(1, 1, 1, -1, 3)
        volume = self._hull_occupancy.float().permute(2, 1, 0)[None, None]
        sampled = F.grid_sample(
            volume,
            sample_grid,
            mode="bilinear",
            padding_mode="zeros",
            align_corners=True,
        )
        return sampled.reshape(-1)

    def _ellipsoid_boundary_model_points(
        self,
        means: torch.Tensor,
        log_scales: torch.Tensor,
        quaternions: torch.Tensor,
        scale_factors: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """Sample the learned oriented ellipsoid at the configured sigma contour."""

        directions = self._ellipsoid_directions.to(means)
        shell_count = max(int(self.config.ellipsoid_radial_shell_count), 1)
        shells = torch.linspace(
            1.0 / float(shell_count),
            1.0,
            shell_count,
            dtype=means.dtype,
            device=means.device,
        )
        probe_directions = (
            directions[None, :, :] * shells[:, None, None]
        ).reshape(-1, 3)
        sigma = torch.exp(log_scales)
        local_offsets = sigma[:, None, :] * probe_directions[None, :, :]
        local_offsets = local_offsets * float(self.config.ellipsoid_sigma_multiplier)
        if scale_factors is not None:
            local_offsets = local_offsets * scale_factors.to(means).reshape(-1, 1, 1)
        rotation = self._quat_wxyz_to_rotation_matrix(quaternions)
        model_offsets = torch.einsum("nij,ndj->ndi", rotation, local_offsets)
        return means[:, None, :] + model_offsets

    def _ellipsoid_boundary_grid_points(
        self,
        means: torch.Tensor,
        log_scales: torch.Tensor,
        quaternions: torch.Tensor,
        scale_factors: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        boundary_model = self._ellipsoid_boundary_model_points(
            means,
            log_scales,
            quaternions,
            scale_factors=scale_factors,
        )
        flat_raw = self._model_points_to_raw(boundary_model.reshape(-1, 3))
        flat_grid = self._raw_points_to_grid(flat_raw)
        return flat_grid.reshape(boundary_model.shape[0], boundary_model.shape[1], 3)

    def _ellipsoid_inside_hull(
        self,
        means: torch.Tensor,
        log_scales: torch.Tensor,
        quaternions: torch.Tensor,
        scale_factors: Optional[torch.Tensor] = None,
    ):
        boundary_grid = self._ellipsoid_boundary_grid_points(
            means,
            log_scales,
            quaternions,
            scale_factors=scale_factors,
        )
        flat_inside, _ = self._lookup_hull(boundary_grid.reshape(-1, 3))
        boundary_inside = flat_inside.reshape(boundary_grid.shape[:2])
        return boundary_inside.all(dim=-1), boundary_inside.float().mean()

    def _maximum_inside_scale_fraction(
        self,
        means: torch.Tensor,
        log_scales: torch.Tensor,
        quaternions: torch.Tensor,
    ) -> torch.Tensor:
        """Find the largest uniform scale whose sampled contour remains in occupied cells."""

        low = torch.zeros(len(means), dtype=means.dtype, device=means.device)
        high = torch.ones_like(low)
        for _ in range(max(int(self.config.ellipsoid_shrink_search_steps), 1)):
            midpoint = 0.5 * (low + high)
            inside, _ = self._ellipsoid_inside_hull(
                means,
                log_scales,
                quaternions,
                scale_factors=midpoint,
            )
            low = torch.where(inside, midpoint, low)
            high = torch.where(inside, high, midpoint)
        return low

    def _capture_center_support(self, allowed_mask: torch.Tensor) -> None:
        if not self.config.gate_densification_by_mask:
            self._last_center_support = None
            return
        means2d = self.info.get("means2d") if isinstance(self.info, dict) else None
        if means2d is None:
            self._last_center_support = None
            return

        xy = means2d.detach()
        if xy.ndim == 3:
            xy = xy[0]
        if xy.ndim != 2 or xy.shape[-1] != 2:
            self._last_center_support = None
            return

        height, width = allowed_mask.shape[:2]
        pixel = torch.round(xy).long()
        valid = (
            torch.isfinite(xy).all(dim=-1)
            & (pixel[:, 0] >= 0)
            & (pixel[:, 0] < width)
            & (pixel[:, 1] >= 0)
            & (pixel[:, 1] < height)
        )
        support = torch.zeros(len(pixel), dtype=torch.bool, device=pixel.device)
        if valid.any():
            support[valid] = allowed_mask[
                pixel[valid, 1], pixel[valid, 0], 0
            ] > 0.5
        self._last_center_support = support

    def _gate_densification_gradients(self) -> None:
        if self._last_center_support is None or not isinstance(self.info, dict):
            return
        means2d = self.info.get("means2d")
        if means2d is None:
            return
        for attribute in ("absgrad", "grad"):
            gradient = getattr(means2d, attribute, None)
            if gradient is None or gradient.ndim < 2:
                continue
            if gradient.shape[-2] != len(self._last_center_support):
                continue
            shape = [1] * gradient.ndim
            shape[-2] = len(self._last_center_support)
            gate = self._last_center_support.to(gradient).view(shape)
            gradient.mul_(gate)

    def _hull_loss_dict(self) -> Dict[str, torch.Tensor]:
        if not self._hull_loaded or self.num_points == 0:
            return {}
        if self.step % max(int(self.config.hull_loss_every), 1) != 0:
            return {}

        sample_count = min(int(self.config.hull_loss_sample_count), self.num_points)
        start = (int(self.step) * 104729) % self.num_points
        index = (start + torch.arange(sample_count, device=self.means.device) * 15485863) % self.num_points
        means = self.means[index]
        raw = self._model_points_to_raw(means)
        grid = self._raw_points_to_grid(raw)
        occupancy = self._sample_hull_occupancy(grid)

        support_loss = torch.relu(float(self.config.hull_min_occupancy) - occupancy).square().mean()
        upper = self._hull_shape.to(grid) - 1.0
        bbox_violation = torch.relu(-grid) + torch.relu(grid - upper)
        bbox_loss = (bbox_violation / torch.clamp(self._hull_shape.to(grid), min=1.0)).square().mean()

        sigma_cap = self._model_voxel_size.to(self.scales) * float(self.config.max_sigma_voxels)
        largest_sigma = torch.exp(self.scales[index]).amax(dim=-1)
        extent_loss = torch.relu(largest_sigma / torch.clamp(sigma_cap, min=1e-8) - 1.0).square().mean()

        losses = {
            "hull_support_loss": support_loss * float(self.config.hull_support_loss_mult),
            "hull_bbox_loss": bbox_loss * float(self.config.hull_bbox_loss_mult),
            "hull_extent_loss": extent_loss * float(self.config.extent_loss_mult),
        }

        if (
            self.config.enable_ellipsoid_constraint
            and self.step % max(int(self.config.ellipsoid_loss_every), 1) == 0
        ):
            ellipsoid_count = min(
                int(self.config.ellipsoid_loss_sample_count), self.num_points
            )
            ellipsoid_index = index[:ellipsoid_count]
            boundary_grid = self._ellipsoid_boundary_grid_points(
                self.means[ellipsoid_index],
                self.scales[ellipsoid_index],
                self.quats[ellipsoid_index],
            )
            boundary_occupancy = self._sample_hull_occupancy(
                boundary_grid.reshape(-1, 3)
            ).reshape(boundary_grid.shape[:2])
            boundary_deficit = torch.relu(
                float(self.config.ellipsoid_min_occupancy) - boundary_occupancy
            )
            mean_deficit = boundary_deficit.square().mean()
            worst_deficit = boundary_deficit.amax(dim=-1).square().mean()

            boundary_upper = self._hull_shape.to(boundary_grid) - 1.0
            boundary_bbox_violation = torch.relu(-boundary_grid) + torch.relu(
                boundary_grid - boundary_upper
            )
            boundary_bbox_loss = (
                boundary_bbox_violation
                / torch.clamp(self._hull_shape.to(boundary_grid), min=1.0)
            ).square().mean()

            losses["hull_ellipsoid_support_loss"] = (
                0.5 * (mean_deficit + worst_deficit)
                * float(self.config.ellipsoid_support_loss_mult)
            )
            losses["hull_ellipsoid_bbox_loss"] = (
                boundary_bbox_loss * float(self.config.ellipsoid_bbox_loss_mult)
            )

        return losses

    def get_loss_dict(self, outputs, batch, metrics_dict=None) -> Dict[str, torch.Tensor]:
        loss_dict = super().get_loss_dict(outputs, batch, metrics_dict)

        if "mask" in batch and "accumulation" in outputs:
            mask = self._downscale_if_required(batch["mask"]).to(self.device).float()
            alpha = outputs["accumulation"].float()
            if mask.ndim == 2:
                mask = mask[..., None]
            if alpha.ndim == 2:
                alpha = alpha[..., None]
            if mask.shape != alpha.shape:
                raise RuntimeError(
                    f"Object mask/accumulation shape mismatch: {tuple(mask.shape)} != {tuple(alpha.shape)}"
                )

            positive = mask > 0.5
            tolerance = max(int(self.config.outside_tolerance_pixels), 0)
            if tolerance > 0:
                kernel = 2 * tolerance + 1
                allowed = F.max_pool2d(
                    mask.permute(2, 0, 1)[None],
                    kernel_size=kernel,
                    stride=1,
                    padding=tolerance,
                )[0].permute(1, 2, 0)
            else:
                allowed = mask
            self._capture_center_support(allowed)

            if int(positive.sum().item()) >= int(self.config.minimum_positive_mask_pixels):
                alpha_safe = alpha.clamp(1e-5, 1.0 - 1e-5)
                negative = allowed < 0.5
                positive_loss = -torch.log(alpha_safe[positive]).mean()
                if negative.any():
                    outside_loss = -torch.log1p(-alpha_safe[negative]).mean()
                else:
                    outside_loss = alpha_safe.new_zeros(())
                intersection = (alpha * mask).sum()
                dice_loss = 1.0 - (2.0 * intersection + 1e-5) / (
                    alpha.sum() + mask.sum() + 1e-5
                )
                loss_dict["alpha_positive_loss"] = (
                    positive_loss * float(self.config.alpha_positive_loss_mult)
                )
                loss_dict["alpha_outside_loss"] = (
                    outside_loss * float(self.config.alpha_outside_loss_mult)
                )
                loss_dict["alpha_dice_loss"] = dice_loss * float(self.config.alpha_dice_loss_mult)
        else:
            self._last_center_support = None

        loss_dict.update(self._hull_loss_dict())
        return loss_dict

    @torch.no_grad()
    def _project_outside_hull(self) -> int:
        """Project escaped centers into occupied support while preserving sub-voxel position."""

        if not self._hull_loaded or self.num_points == 0:
            return 0
        raw = self._model_points_to_raw(self.means.data)
        grid = self._raw_points_to_grid(raw)
        inside, clamped = self._lookup_hull(grid)
        outside = ~inside

        if outside.any():
            nearest = self._hull_nearest[
                clamped[outside, 0], clamped[outside, 1], clamped[outside, 2]
            ].to(raw)
            rounded_local = torch.round(grid[outside])
            residual = torch.clamp(grid[outside] - rounded_local, -0.40, 0.40)
            projected_local = (nearest + residual) * self._hull_voxel_size.to(raw)
            projected_raw = self._hull_origin.to(raw) + projected_local @ self._hull_axes.to(raw).T
            self.means.data[outside] = self._raw_points_to_model(projected_raw)
        return int(outside.sum().item())

    @torch.no_grad()
    def _snap_centers_to_occupied_cell(self, indices: torch.Tensor) -> None:
        """Move severe boundary cases to the center of their current occupied cell."""

        if not len(indices):
            return
        raw = self._model_points_to_raw(self.means.data[indices])
        grid = self._raw_points_to_grid(raw)
        inside, clamped = self._lookup_hull(grid)
        if not inside.all():
            raise RuntimeError("Ellipsoid correction received a center outside the support hull.")
        snapped_local = clamped.to(raw) * self._hull_voxel_size.to(raw)
        snapped_raw = self._hull_origin.to(raw) + snapped_local @ self._hull_axes.to(raw).T
        self.means.data[indices] = self._raw_points_to_model(snapped_raw)

    @torch.no_grad()
    def _constrain_ellipsoid_extents(self) -> Dict[str, float]:
        """Shrink only Gaussians whose sampled oriented contour crosses the support volume."""

        if (
            not self._hull_loaded
            or not self.config.enable_ellipsoid_constraint
            or not self.config.hard_constrain_ellipsoid
            or self.num_points == 0
        ):
            return {"shrunk": 0, "snapped": 0, "boundary_inside_fraction": 1.0}

        shrunk_total = 0
        snapped_total = 0
        inside_sample_total = 0.0
        boundary_sample_total = 0
        chunk_size = max(int(self.config.ellipsoid_hard_chunk_size), 1)

        for start in range(0, self.num_points, chunk_size):
            end = min(start + chunk_size, self.num_points)
            chunk_index = torch.arange(start, end, device=self.means.device)
            means = self.means.data[chunk_index]
            scales = self.scales.data[chunk_index]
            quats = self.quats.data[chunk_index]
            inside, _ = self._ellipsoid_inside_hull(means, scales, quats)
            violating_local = torch.nonzero(~inside, as_tuple=False).reshape(-1)
            if len(violating_local):
                violating_index = chunk_index[violating_local]
                factors = self._maximum_inside_scale_fraction(
                    self.means.data[violating_index],
                    self.scales.data[violating_index],
                    self.quats.data[violating_index],
                )
                severe = factors < float(self.config.ellipsoid_snap_threshold)
                snapped_mask = severe.clone()
                if severe.any():
                    severe_index = violating_index[severe]
                    self._snap_centers_to_occupied_cell(severe_index)
                    factors[severe] = self._maximum_inside_scale_fraction(
                        self.means.data[severe_index],
                        self.scales.data[severe_index],
                        self.quats.data[severe_index],
                    )
                    snapped_total += int(severe.sum().item())

                # Stay just inside the last passing binary-search contour.
                factors = torch.clamp(factors * 0.98, min=1e-4, max=1.0)
                for _ in range(8):
                    corrected_inside, _ = self._ellipsoid_inside_hull(
                        self.means.data[violating_index],
                        self.scales.data[violating_index],
                        self.quats.data[violating_index],
                        scale_factors=factors,
                    )
                    if corrected_inside.all():
                        break
                    factors[~corrected_inside] = torch.clamp(
                        factors[~corrected_inside] * 0.5,
                        min=1e-4,
                    )
                corrected_inside, _ = self._ellipsoid_inside_hull(
                    self.means.data[violating_index],
                    self.scales.data[violating_index],
                    self.quats.data[violating_index],
                    scale_factors=factors,
                )
                if not corrected_inside.all():
                    newly_snapped = ~corrected_inside & ~snapped_mask
                    if newly_snapped.any():
                        self._snap_centers_to_occupied_cell(violating_index[newly_snapped])
                        snapped_total += int(newly_snapped.sum().item())
                    factors[~corrected_inside] = 1e-4
                self.scales.data[violating_index] += torch.log(factors)[:, None]
                shrunk_total += int(len(violating_index))

            _, boundary_fraction = self._ellipsoid_inside_hull(
                self.means.data[chunk_index],
                self.scales.data[chunk_index],
                self.quats.data[chunk_index],
            )
            probe_count = int(self._ellipsoid_directions.shape[0]) * max(
                int(self.config.ellipsoid_radial_shell_count), 1
            )
            inside_sample_total += float(boundary_fraction.item()) * len(chunk_index) * probe_count
            boundary_sample_total += len(chunk_index) * probe_count

        boundary_inside_fraction = (
            inside_sample_total / float(boundary_sample_total)
            if boundary_sample_total
            else 1.0
        )
        return {
            "shrunk": shrunk_total,
            "snapped": snapped_total,
            "boundary_inside_fraction": boundary_inside_fraction,
        }

    @torch.no_grad()
    def _enforce_hull_containment(self) -> Dict[str, float]:
        center_projected = self._project_outside_hull()
        sigma_cap = self._model_voxel_size.to(self.scales) * float(self.config.max_sigma_voxels)
        self.scales.data.clamp_(max=torch.log(torch.clamp(sigma_cap, min=1e-8)))
        extent_stats = self._constrain_ellipsoid_extents()
        return {
            "center_projected": center_projected,
            **extent_stats,
        }

    def step_post_backward(self, step):
        self._gate_densification_gradients()
        super().step_post_backward(step)

        containment_stats = {
            "center_projected": 0,
            "shrunk": 0,
            "snapped": 0,
            "boundary_inside_fraction": self._last_boundary_inside_fraction,
        }
        if (
            self._hull_loaded
            and self.config.hard_project_outside_hull
            and step % max(int(self.config.hull_projection_interval), 1) == 0
        ):
            containment_stats = self._enforce_hull_containment()
        self._last_projected_count = int(containment_stats["center_projected"])
        self._last_ellipsoid_shrunk_count = int(containment_stats["shrunk"])
        self._last_ellipsoid_snapped_count = int(containment_stats["snapped"])
        self._last_boundary_inside_fraction = float(
            containment_stats["boundary_inside_fraction"]
        )
        self._total_projected_count += self._last_projected_count
        self._total_ellipsoid_shrunk_count += self._last_ellipsoid_shrunk_count
        self._total_ellipsoid_snapped_count += self._last_ellipsoid_snapped_count

        if self._hull_loaded and step % max(int(self.config.projection_log_interval), 1) == 0:
            print(
                f"[object-hull] step={step} center_projected={self._last_projected_count} "
                f"total_center_projected={self._total_projected_count} "
                f"ellipsoid_shrunk={self._last_ellipsoid_shrunk_count} "
                f"total_ellipsoid_shrunk={self._total_ellipsoid_shrunk_count} "
                f"ellipsoid_snapped={self._last_ellipsoid_snapped_count} "
                f"boundary_inside_fraction={self._last_boundary_inside_fraction:.6f} "
                f"gaussians={self.num_points}"
            )

    def get_metrics_dict(self, outputs, batch) -> Dict[str, torch.Tensor]:
        metrics = super().get_metrics_dict(outputs, batch)
        if self._hull_loaded and self.num_points:
            with torch.no_grad():
                sample_count = min(8192, self.num_points)
                raw = self._model_points_to_raw(self.means[:sample_count])
                inside, _ = self._lookup_hull(self._raw_points_to_grid(raw))
                metrics["hull_inside_fraction"] = inside.float().mean()
                metrics["hull_center_inside_fraction"] = inside.float().mean()
                metrics["hull_projected_last"] = torch.tensor(
                    float(self._last_projected_count), device=self.device
                )
                if self.config.enable_ellipsoid_constraint:
                    ellipsoid_count = min(1024, self.num_points)
                    ellipsoid_inside, boundary_fraction = self._ellipsoid_inside_hull(
                        self.means[:ellipsoid_count],
                        self.scales[:ellipsoid_count],
                        self.quats[:ellipsoid_count],
                    )
                    metrics["hull_ellipsoid_inside_fraction"] = (
                        ellipsoid_inside.float().mean()
                    )
                    metrics["hull_boundary_sample_inside_fraction"] = boundary_fraction
                    metrics["hull_ellipsoid_shrunk_last"] = torch.tensor(
                        float(self._last_ellipsoid_shrunk_count), device=self.device
                    )
                    metrics["hull_ellipsoid_snapped_last"] = torch.tensor(
                        float(self._last_ellipsoid_snapped_count), device=self.device
                    )
        return metrics


def _build_method_spec(
    method_name: str,
    enable_hull: bool,
    enable_ellipsoid: bool = False,
) -> MethodSpecification:
    trainer = copy.deepcopy(method_configs["splatfacto"])
    trainer.method_name = method_name
    trainer.pipeline.model = ObjectConstrainedSplatfactoModelConfig(
        enable_hull_constraint=enable_hull,
        hard_project_outside_hull=enable_hull,
        enable_ellipsoid_constraint=enable_ellipsoid,
        hard_constrain_ellipsoid=enable_ellipsoid,
        hull_support_loss_mult=0.05 if enable_hull else 0.0,
        hull_bbox_loss_mult=0.10 if enable_hull else 0.0,
        extent_loss_mult=0.05 if enable_hull else 0.0,
        ellipsoid_support_loss_mult=0.10 if enable_ellipsoid else 0.0,
        ellipsoid_bbox_loss_mult=0.10 if enable_ellipsoid else 0.0,
        use_scale_regularization=True,
        max_gauss_ratio=5.0,
        background_color="random",
    )
    if enable_ellipsoid:
        suffix = "and an oriented finite-sigma ellipsoid constrained to a multi-view support hull."
    elif enable_hull:
        suffix = "and a center-only multi-view support hull."
    else:
        suffix = "and no 3D hull constraint."
    description = (
        "Splatfacto with explicit object-opacity supervision, silhouette-gated densification, "
        + suffix
    )
    return MethodSpecification(config=trainer, description=description)


ObjectAlphaSplatfacto = _build_method_spec("object-alpha-splatfacto", enable_hull=False)
ObjectHullSplatfacto = _build_method_spec("object-hull-splatfacto", enable_hull=True)
ObjectEllipsoidSplatfacto = _build_method_spec(
    "object-ellipsoid-splatfacto",
    enable_hull=True,
    enable_ellipsoid=True,
)
'''

PLUGIN_DIR = Path('/content/room-model-object-splat-plugin')
PLUGIN_DIR.mkdir(parents=True, exist_ok=True)
PLUGIN_PATH = PLUGIN_DIR / 'object_constrained_splatfacto.py'
PLUGIN_PATH.write_text(OBJECT_CONSTRAINED_PLUGIN_SOURCE, encoding='utf-8')
PLUGIN_SHA256 = hashlib.sha256(OBJECT_CONSTRAINED_PLUGIN_SOURCE.encode('utf-8')).hexdigest()
CUSTOM_METHOD_DEFINITIONS = (
    'object-alpha-splatfacto=object_constrained_splatfacto:ObjectAlphaSplatfacto,'
    'object-hull-splatfacto=object_constrained_splatfacto:ObjectHullSplatfacto,'
    'object-ellipsoid-splatfacto=object_constrained_splatfacto:ObjectEllipsoidSplatfacto'
)

def nerfstudio_subprocess_env():
    environment = os.environ.copy()
    existing_pythonpath = environment.get('PYTHONPATH', '')
    environment['PYTHONPATH'] = str(PLUGIN_DIR) + (os.pathsep + existing_pythonpath if existing_pythonpath else '')
    environment['NERFSTUDIO_METHOD_CONFIGS'] = CUSTOM_METHOD_DEFINITIONS
    environment['MAX_JOBS'] = '1'
    try:
        if torch.cuda.is_available():
            major, minor = torch.cuda.get_device_capability(0)
            environment['TORCH_CUDA_ARCH_LIST'] = f'{major}.{minor}'
    except Exception:
        pass
    return environment

for method_name in [
    'object-alpha-splatfacto',
    'object-hull-splatfacto',
    'object-ellipsoid-splatfacto',
]:
    result = subprocess.run(
        ['ns-train', method_name, '--help'],
        env=nerfstudio_subprocess_env(),
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print(result.stdout[-4000:])
        print(result.stderr[-4000:])
        fail(f'Embedded Nerfstudio method registration failed: {method_name}')
    print('Registered method:', method_name)
print('Plugin path:', PLUGIN_PATH)
print('Plugin SHA-256:', PLUGIN_SHA256)


## 4. Utility Functions

In [ ]:
def load_json(path):
    with Path(path).open('r') as f:
        return json.load(f)


def stable_json_hash(data):
    payload = json.dumps(data, sort_keys=True, separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def file_inventory_hash(paths):
    rows = []
    for path in sorted(Path(p) for p in paths):
        stat = path.stat()
        rows.append((path.name, int(stat.st_size), int(stat.st_mtime_ns)))
    return stable_json_hash(rows)


def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2))


def safe_write_csv(path, rows, columns=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    records = list(rows)
    if columns is None:
        columns = []
        for record in records:
            for key in record.keys():
                if key not in columns:
                    columns.append(key)
    with path.open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(columns or []))
        writer.writeheader()
        for record in records:
            writer.writerow({key: record.get(key, '') for key in columns})


def safe_read_csv(path):
    with Path(path).open('r', newline='') as f:
        return list(csv.DictReader(f))


def print_rows(rows, columns=None, limit=20):
    rows = list(rows)
    if columns is None:
        columns = []
        for row in rows[:limit]:
            for key in row.keys():
                if key not in columns:
                    columns.append(key)
    if not rows:
        print('(no rows)')
        return
    shown = rows[:limit]
    widths = {col: max(len(str(col)), *(len(str(row.get(col, ''))) for row in shown)) for col in columns}
    print(' | '.join(str(col).ljust(widths[col]) for col in columns))
    print('-+-'.join('-' * widths[col] for col in columns))
    for row in shown:
        print(' | '.join(str(row.get(col, '')).ljust(widths[col]) for col in columns))
    if len(rows) > limit:
        print(f'... {len(rows) - limit} more rows')


def describe_numeric(values):
    clean = []
    for value in values:
        try:
            number = float(value)
        except Exception:
            continue
        if np.isfinite(number):
            clean.append(number)
    if not clean:
        return {'count': 0}
    arr = np.asarray(clean, dtype=np.float64)
    return {
        'count': int(arr.size),
        'mean': float(arr.mean()),
        'std': float(arr.std()),
        'min': float(arr.min()),
        'p25': float(np.percentile(arr, 25)),
        'median': float(np.median(arr)),
        'p75': float(np.percentile(arr, 75)),
        'max': float(arr.max()),
    }


def normalize_label(label):
    return re.sub(r'[^a-z0-9]+', '_', str(label).strip().lower()).strip('_')



def object_key_for_instance(label_norm, instance_id):
    label_norm = normalize_label(label_norm)
    instance_norm = normalize_label(instance_id)
    if not instance_norm:
        raise ValueError(f'Empty instance ID for label {label_norm!r}.')
    if instance_norm == label_norm or instance_norm.startswith(label_norm + '_'):
        return instance_norm
    return f'{label_norm}__{instance_norm}'


def target_semantic_label(target_key):
    if target_key not in TARGET_OBJECT_LABELS:
        raise KeyError(f'Unknown target object key: {target_key!r}')
    return TARGET_OBJECT_LABELS[target_key]


def target_instance_id(target_key):
    if target_key not in RESOLVED_INSTANCE_IDS:
        raise KeyError(f'Unknown target object key: {target_key!r}')
    return RESOLVED_INSTANCE_IDS[target_key]


def configured_instance_ids(label_norm):
    configured = TARGET_INSTANCE_IDS.get(label_norm)
    if configured is None:
        return None
    if isinstance(configured, (str, int)):
        return {str(configured)}
    return {str(value) for value in configured}


def frame_key_from_any(value):
    stem = Path(str(value)).stem
    stem = re.sub(r'(_rgb|_image|_mask|_overlay)$', '', stem)
    return stem


def resolve_scan_path(value, base_dirs):
    if value is None:
        return None
    candidate = Path(str(value))
    if candidate.is_absolute() and candidate.exists():
        return candidate
    for base in base_dirs:
        path = Path(base) / candidate
        if path.exists():
            return path
    return candidate if candidate.is_absolute() else None


def frame_image_path(frame):
    file_path = frame.get('file_path')
    if not file_path:
        return None
    candidate = Path(file_path)
    if candidate.is_absolute() and candidate.exists():
        return candidate
    candidates = [
        SOURCE_NS_DATA_DIR / file_path,
        SOURCE_NS_DATA_DIR / 'images' / Path(file_path).name,
        SCAN_DIR / file_path,
    ]
    for path in candidates:
        if path.exists():
            return path
    return candidates[0]


def frame_intrinsics(frame, transforms_data, image_shape):
    h, w = image_shape[:2]
    fx = frame.get('fl_x', transforms_data.get('fl_x'))
    fy = frame.get('fl_y', transforms_data.get('fl_y', fx))
    cx = frame.get('cx', transforms_data.get('cx', w / 2.0))
    cy = frame.get('cy', transforms_data.get('cy', h / 2.0))
    if fx is None:
        camera_angle_x = frame.get('camera_angle_x', transforms_data.get('camera_angle_x'))
        if camera_angle_x is None:
            fail('Missing fl_x and camera_angle_x in transforms.json')
        fx = 0.5 * w / math.tan(0.5 * float(camera_angle_x))
    if fy is None:
        fy = fx
    return float(fx), float(fy), float(cx), float(cy)


def mask_dilation_for_label(label_norm):
    return int(MASK_DILATION_PIXELS_BY_LABEL.get(label_norm, MASK_DILATION_PIXELS))


def mask_erosion_for_label(label_norm):
    return int(MASK_EROSION_PIXELS_BY_LABEL.get(label_norm, MASK_EROSION_PIXELS))


def train_iterations_for_label(label_norm):
    if TRAINING_PROFILE != 'final':
        return int(TRAIN_ITERATIONS)
    return int(TRAIN_ITERATIONS_BY_LABEL.get(label_norm, TRAIN_ITERATIONS))


def read_mask(mask_path, image_shape, label_norm=None):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(f'Could not read mask: {mask_path}')
    h, w = image_shape[:2]
    if mask.shape[:2] != (h, w):
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
    mask = (mask > 0).astype(np.uint8)
    dilation_pixels = mask_dilation_for_label(label_norm) if label_norm is not None else int(MASK_DILATION_PIXELS)
    erosion_pixels = mask_erosion_for_label(label_norm) if label_norm is not None else int(MASK_EROSION_PIXELS)
    if dilation_pixels > 0:
        kernel = np.ones((2 * int(dilation_pixels) + 1, 2 * int(dilation_pixels) + 1), dtype=np.uint8)
        mask = cv2.dilate(mask, kernel, iterations=1)
    if erosion_pixels > 0:
        kernel = np.ones((2 * int(erosion_pixels) + 1, 2 * int(erosion_pixels) + 1), dtype=np.uint8)
        mask = cv2.erode(mask, kernel, iterations=1)
    return mask.astype(bool)


def suffixed_dataset_dir(base_dir, factor):
    base_dir = Path(base_dir)
    if int(factor) == 1:
        return base_dir
    return base_dir.parent / f'{base_dir.name}_{int(factor)}'


def label_paths(target_key):
    target_key = normalize_label(target_key)
    label_norm = target_semantic_label(target_key)
    instance_id = target_instance_id(target_key)
    run_id = f'{target_key}_{BATCH_ID}'
    root = BATCH_ROOT / target_key
    local_root = LOCAL_CACHE_ROOT / BATCH_ID / target_key
    local_ns_data = local_root / 'nerfstudio-data'
    return {
        'label': label_norm,
        'object_key': target_key,
        'instance_id': instance_id,
        'run_id': run_id,
        'root': root,
        'ns_data': local_ns_data,
        'local_root': local_root,
        'image_dir': local_ns_data / 'images',
        'mask_dir': local_ns_data / 'masks',
        'preview_dir': local_root / 'masked_rgb_preview',
        'training_dir': root / 'training',
        'export_dir': root / 'exports',
        'alignment_manifest': root / 'object_splat_alignment_manifest.json',
        'report_dir': REPORT_DIR / target_key,
        'dataset_stage_manifest': root / 'dataset_stage_manifest.json',
    }


def ensure_symlink(link_path, target_path):
    link_path = Path(link_path)
    target_path = Path(target_path)
    if link_path.is_symlink() and link_path.resolve() == target_path.resolve():
        return
    if link_path.exists() or link_path.is_symlink():
        if link_path.is_dir() and not link_path.is_symlink():
            shutil.rmtree(link_path)
        else:
            link_path.unlink()
    link_path.parent.mkdir(parents=True, exist_ok=True)
    link_path.symlink_to(target_path, target_is_directory=True)


def shared_image_dir(factor):
    return suffixed_dataset_dir(LOCAL_SHARED_DATA_DIR / 'images', factor)


def shared_cached_image_path(frame, factor=1):
    source_path = frame_image_path(frame)
    if source_path is None:
        return None
    return shared_image_dir(factor) / Path(source_path).name


def prepare_shared_local_image_cache():
    source_images = [frame_image_path(frame) for frame in frames]
    missing = [str(path) for path in source_images if path is None or not path.exists()]
    if missing:
        raise RuntimeError(f'{len(missing)} source images are missing before shared-cache creation.')
    cache_manifest_path = LOCAL_SHARED_DATA_DIR / 'shared_cache_manifest.json'
    fingerprint = stable_json_hash({
        'version': BATCH_PIPELINE_VERSION,
        'source_transforms_hash': stable_json_hash(transforms_data),
        'source_images': file_inventory_hash(source_images),
        'downscale_factors': [int(value) for value in DOWNSCALE_FACTORS_TO_WRITE],
    })
    if REUSE_LOCAL_SHARED_CACHE and cache_manifest_path.exists():
        previous = load_json(cache_manifest_path)
        complete = all(
            shared_cached_image_path(frame, factor).exists()
            for frame in frames
            for factor in DOWNSCALE_FACTORS_TO_WRITE
        )
        if previous.get('fingerprint') == fingerprint and complete:
            print('Reusing shared local image cache:', LOCAL_SHARED_DATA_DIR)
            return previous

    if LOCAL_SHARED_DATA_DIR.exists():
        shutil.rmtree(LOCAL_SHARED_DATA_DIR)
    for factor in DOWNSCALE_FACTORS_TO_WRITE:
        shared_image_dir(factor).mkdir(parents=True, exist_ok=True)

    for frame, source_path in tqdm(list(zip(frames, source_images)), desc='Build shared RGB cache'):
        image = None
        for factor in DOWNSCALE_FACTORS_TO_WRITE:
            factor = int(factor)
            output_path = shared_cached_image_path(frame, factor)
            if factor == 1:
                shutil.copy2(source_path, output_path)
                continue
            if image is None:
                image = cv2.imread(str(source_path), cv2.IMREAD_COLOR)
                if image is None:
                    raise RuntimeError(f'Could not read source image for cache: {source_path}')
            h, w = image.shape[:2]
            resized = cv2.resize(
                image,
                (max(1, int(round(w / factor))), max(1, int(round(h / factor)))),
                interpolation=cv2.INTER_AREA,
            )
            if not cv2.imwrite(str(output_path), resized):
                raise RuntimeError(f'Failed to write shared cached image: {output_path}')

    manifest = {
        'schema': 'object_splat_shared_cache.v1',
        'fingerprint': fingerprint,
        'created_at': datetime.now().isoformat(timespec='seconds'),
        'source_transforms': str(TRANSFORMS_PATH),
        'frame_count': len(frames),
        'downscale_factors': [int(value) for value in DOWNSCALE_FACTORS_TO_WRITE],
        'local_cache': str(LOCAL_SHARED_DATA_DIR),
    }
    write_json(cache_manifest_path, manifest)
    print('Built shared local image cache:', LOCAL_SHARED_DATA_DIR)
    return manifest


def link_shared_images_into_label_dataset(paths):
    for factor in DOWNSCALE_FACTORS_TO_WRITE:
        ensure_symlink(suffixed_dataset_dir(paths['image_dir'], factor), shared_image_dir(factor))


def write_dataset_masks(paths, mask, out_mask_name):
    h, w = mask.shape[:2]
    written = []
    for factor in DOWNSCALE_FACTORS_TO_WRITE:
        factor = int(factor)
        mask_dir = suffixed_dataset_dir(paths['mask_dir'], factor)
        mask_dir.mkdir(parents=True, exist_ok=True)
        out_mask_path = mask_dir / out_mask_name
        if factor == 1:
            mask_to_write = mask.astype(np.uint8)
        else:
            mask_to_write = cv2.resize(
                mask.astype(np.uint8),
                (max(1, int(round(w / factor))), max(1, int(round(h / factor)))),
                interpolation=cv2.INTER_NEAREST,
            )
        if not cv2.imwrite(str(out_mask_path), mask_to_write * 255):
            raise RuntimeError(f'Failed to write dataset mask: {out_mask_path}')
        written.append({'factor': factor, 'mask_path': str(out_mask_path)})
    return written

def iter_detection_items(data):
    if isinstance(data, list):
        for item in data:
            if isinstance(item, dict):
                yield item
        return
    if not isinstance(data, dict):
        return
    for key in ['detections', 'objects', 'masks', 'segments', 'instances']:
        items = data.get(key)
        if isinstance(items, list):
            for item in items:
                if isinstance(item, dict):
                    yield item


def detection_label(item):
    for key in ['label', 'class', 'class_name', 'name', 'category']:
        if item.get(key):
            return normalize_label(item[key])
    return None


def detection_mask_path(item):
    for key in ['mask_path', 'mask_file', 'mask', 'segmentation_mask_path', 'path']:
        if item.get(key):
            return item[key]
    return None


def detection_object_id(item):
    for key in ['object_id', 'id', 'instance_id', 'track_id']:
        if item.get(key) is not None:
            return str(item[key])
    return None


def detection_record_is_authoritative(item, target_label_norm):
    # Canonical v2 JSON contains accepted masks only, but these checks make the batch
    # fail closed if a hand-edited or older file introduces rejected/unknown records.
    if item.get('accepted') is not True or item.get('state') != 'visible':
        return False
    role = item.get('segmentation_role')
    source = item.get('mask_source')
    if target_label_norm in {'wall', 'floor', 'ceiling'}:
        return role == 'semantic_room_structure' and source == 'segformer_ade_structural'
    return role == 'tracked_object' and source == 'grounding_dino_sam2_video'


def collect_target_masks_from_detections(target_label_norm, target_object_id=None):
    frame_to_masks = {}
    rows = []
    json_paths = sorted(DETECTION_DIR.glob('*.json')) if DETECTION_DIR.exists() else []
    base_dirs = [PROJECT_ROOT, SCAN_DIR, SOURCE_NS_DATA_DIR, DETECTION_DIR, *MASK_DIRS]
    for json_path in json_paths:
        try:
            data = load_json(json_path)
        except Exception as exc:
            logger.warning('Skipping unreadable detection JSON %s: %s', json_path, exc)
            continue
        if REQUIRE_TRACKED_SEGMENTATION_V2 and data.get('schema') != 'tracked_semantic_frame.v2':
            fail(
                f'Expected tracked_semantic_frame.v2 in {json_path}, found {data.get("schema")!r}. '
                'Run the current segmentation.ipynb through its audit stage.'
            )
        default_frame = data.get('frame') or data.get('image') or data.get('image_path') or json_path.stem
        for item in iter_detection_items(data):
            label = detection_label(item)
            object_id = detection_object_id(item)
            if label != target_label_norm:
                continue
            if target_object_id is not None and object_id != str(target_object_id):
                continue
            if not detection_record_is_authoritative(item, target_label_norm):
                logger.warning('Ignoring non-authoritative mask record for %s in %s', label, json_path)
                continue
            mask_path = resolve_scan_path(detection_mask_path(item), base_dirs)
            if mask_path is None or not mask_path.exists():
                logger.warning('Detection mask missing for %s in %s: %s', label, json_path, detection_mask_path(item))
                continue
            item_frame = item.get('frame') or item.get('image') or item.get('image_path') or default_frame
            frame_key = frame_key_from_any(item_frame)
            frame_to_masks.setdefault(frame_key, []).append(mask_path)
            rows.append({
                'frame_key': frame_key,
                'label': label,
                'object_id': object_id,
                'mask_path': str(mask_path),
                'source': str(json_path),
                'segmentation_fingerprint': data.get('segmentation_fingerprint'),
                'layout_fingerprint': data.get('layout_fingerprint'),
                'segmentation_role': item.get('segmentation_role'),
                'mask_source': item.get('mask_source'),
            })
    return frame_to_masks, rows


def collect_target_masks_by_filename(target_label_norm, frame_keys):
    frame_to_masks = {}
    rows = []
    label_token = target_label_norm
    for mask_dir in MASK_DIRS:
        if not mask_dir.exists():
            continue
        for mask_path in sorted(mask_dir.glob('*.png')):
            stem_norm = normalize_label(mask_path.stem)
            parts = stem_norm.split('_')
            if label_token not in parts and not stem_norm.endswith('_' + label_token) and f'_{label_token}_' not in f'_{stem_norm}_':
                continue
            matched_frame_key = None
            for frame_key in frame_keys:
                if mask_path.stem.startswith(frame_key):
                    matched_frame_key = frame_key
                    break
            if matched_frame_key is None:
                match = re.match(r'(frame_\d+)', mask_path.stem)
                matched_frame_key = match.group(1) if match else mask_path.stem
            frame_to_masks.setdefault(matched_frame_key, []).append(mask_path)
            rows.append({
                'frame_key': matched_frame_key,
                'label': target_label_norm,
                'object_id': None,
                'mask_path': str(mask_path),
                'source': 'filename_scan',
            })
    return frame_to_masks, rows


def merge_mask_maps(*maps):
    merged = {}
    seen = set()
    for mapping in maps:
        for frame_key, masks in mapping.items():
            for mask_path in masks:
                key = (frame_key, str(mask_path))
                if key in seen:
                    continue
                seen.add(key)
                merged.setdefault(frame_key, []).append(Path(mask_path))
    return merged


def summarize_instance_candidates(rows):
    grouped = {}
    for row in rows:
        object_id = row.get('object_id')
        if not object_id:
            continue
        entry = grouped.setdefault(str(object_id), {'frames': set(), 'mask_records': 0})
        entry['frames'].add(row.get('frame_key'))
        entry['mask_records'] += 1
    return [
        {
            'instance_id': instance_id,
            'visible_frames': len(values['frames']),
            'mask_records': int(values['mask_records']),
        }
        for instance_id, values in sorted(grouped.items())
    ]


def resolve_target_instance(label_norm, rows):
    candidates = summarize_instance_candidates(rows)
    if not candidates:
        return None, []
    configured = TARGET_INSTANCE_IDS.get(label_norm)
    if configured is not None:
        selected = next(
            (row for row in candidates if row['instance_id'] == str(configured)),
            None,
        )
        if selected is None:
            raise RuntimeError(
                f'Configured instance {configured!r} for {label_norm!r} was not found. '
                'Available: ' + ', '.join(row['instance_id'] for row in candidates)
            )
    else:
        selected = sorted(
            candidates,
            key=lambda row: (
                -row['visible_frames'],
                -row['mask_records'],
                row['instance_id'],
            ),
        )[0]
    for row in candidates:
        row['selected'] = row['instance_id'] == selected['instance_id']
        row['selection_policy'] = INSTANCE_SELECTION_POLICY
    return selected['instance_id'], candidates


def pilot_variant_paths(base_paths, variant_name, object_sparse_path=None, support_grid_path=None):
    if variant_name not in PILOT_VARIANTS:
        raise KeyError(f'Unknown pilot variant: {variant_name}')
    variant = PILOT_VARIANTS[variant_name]
    result = dict(base_paths)
    variant_root = base_paths['root'] / 'variants' / variant_name
    variant_local_root = base_paths['local_root'] / 'variants' / variant_name
    result.update({
        'variant_name': variant_name,
        'method_name': variant['method'],
        'variant_description': variant['description'],
        'root': variant_root,
        'local_root': variant_local_root,
        'training_dir': variant_root / 'training',
        'export_dir': variant_root / 'exports',
        'report_dir': base_paths['report_dir'] / 'variants' / variant_name,
        'run_id': f"{base_paths['object_key']}_{variant_name}_{BATCH_ID}",
        'model_cli_args': [],
    })
    if variant_name != 'masked_baseline':
        result['model_cli_args'].extend([
            '--pipeline.model.alpha-positive-loss-mult', str(PILOT_ALPHA_POSITIVE_LOSS_MULT),
            '--pipeline.model.alpha-outside-loss-mult', str(PILOT_ALPHA_OUTSIDE_LOSS_MULT),
            '--pipeline.model.alpha-dice-loss-mult', str(PILOT_ALPHA_DICE_LOSS_MULT),
            '--pipeline.model.outside-tolerance-pixels', str(PILOT_OUTSIDE_TOLERANCE_PIXELS),
        ])
    if variant_name == 'ellipsoid_constrained':
        if not object_sparse_path or not support_grid_path:
            raise RuntimeError('The ellipsoid-constrained arm requires sparse seed and support-grid paths.')
        result['model_cli_args'].extend([
            '--pipeline.model.raw-seed-path', str(object_sparse_path),
            '--pipeline.model.support-grid-path', str(support_grid_path),
            '--pipeline.model.max-sigma-voxels', str(PILOT_MAX_SIGMA_VOXELS),
            '--pipeline.model.hull-support-loss-mult', str(PILOT_HULL_SUPPORT_LOSS_MULT),
            '--pipeline.model.hull-bbox-loss-mult', str(PILOT_HULL_BBOX_LOSS_MULT),
            '--pipeline.model.extent-loss-mult', str(PILOT_HULL_EXTENT_LOSS_MULT),
            '--pipeline.model.hull-projection-interval', str(PILOT_HULL_PROJECTION_INTERVAL),
            '--pipeline.model.ellipsoid-sigma-multiplier', str(PILOT_ELLIPSOID_SIGMA_MULTIPLIER),
            '--pipeline.model.ellipsoid-fibonacci-direction-count', str(PILOT_ELLIPSOID_FIBONACCI_DIRECTION_COUNT),
            '--pipeline.model.ellipsoid-radial-shell-count', str(PILOT_ELLIPSOID_RADIAL_SHELL_COUNT),
            '--pipeline.model.ellipsoid-support-loss-mult', str(PILOT_ELLIPSOID_SUPPORT_LOSS_MULT),
            '--pipeline.model.ellipsoid-bbox-loss-mult', str(PILOT_ELLIPSOID_BBOX_LOSS_MULT),
            '--pipeline.model.ellipsoid-min-occupancy', str(PILOT_ELLIPSOID_MIN_OCCUPANCY),
            '--pipeline.model.ellipsoid-loss-sample-count', str(PILOT_ELLIPSOID_LOSS_SAMPLE_COUNT),
            '--pipeline.model.ellipsoid-loss-every', str(PILOT_ELLIPSOID_LOSS_EVERY),
            '--pipeline.model.ellipsoid-hard-chunk-size', str(PILOT_ELLIPSOID_HARD_CHUNK_SIZE),
            '--pipeline.model.ellipsoid-shrink-search-steps', str(PILOT_ELLIPSOID_SHRINK_SEARCH_STEPS),
            '--pipeline.model.ellipsoid-snap-threshold', str(PILOT_ELLIPSOID_SNAP_THRESHOLD),
        ])
    return result


## 5. Load Source Dataset And Audit Labels

In [ ]:
TRANSFORMS_PATH = SOURCE_NS_DATA_DIR / 'transforms.json'

if not PROJECT_ROOT.exists():
    fail(f'Missing project root: {PROJECT_ROOT}')
if not SCAN_DIR.exists():
    fail(f'Missing scan dir: {SCAN_DIR}')
if not TRANSFORMS_PATH.exists():
    fail(f'Missing transforms.json: {TRANSFORMS_PATH}. Run reconstruction.ipynb first.')

detection_json_paths = sorted(DETECTION_DIR.glob('*.json'))
if REQUIRE_TRACKED_SEGMENTATION_V2 and not detection_json_paths:
    fail(f'No tracked detection JSON files found in {DETECTION_DIR}. Run segmentation.ipynb first.')
detection_payloads = [(path, load_json(path)) for path in detection_json_paths]
invalid_detection_files = [
    str(path) for path, data in detection_payloads
    if data.get('schema') != 'tracked_semantic_frame.v2'
    or not data.get('segmentation_fingerprint')
    or not data.get('layout_fingerprint')
]
if REQUIRE_TRACKED_SEGMENTATION_V2 and invalid_detection_files:
    fail(
        f'{len(invalid_detection_files)} detection JSON files are stale or incomplete. '
        'Run the current segmentation.ipynb through its audit stage. First invalid file: '
        + invalid_detection_files[0]
    )
segmentation_fingerprints = {
    data.get('segmentation_fingerprint') for _, data in detection_payloads
}
layout_fingerprints = {data.get('layout_fingerprint') for _, data in detection_payloads}
if REQUIRE_TRACKED_SEGMENTATION_V2 and (
    len(segmentation_fingerprints) != 1 or len(layout_fingerprints) != 1
):
    fail(
        'Expected one tracked segmentation fingerprint and one layout fingerprint; found '
        f'{sorted(segmentation_fingerprints)} and {sorted(layout_fingerprints)}. '
        'Rerun segmentation.ipynb through Audit.'
    )
SEGMENTATION_FINGERPRINT = next(iter(segmentation_fingerprints), None)
LAYOUT_FINGERPRINT = next(iter(layout_fingerprints), None)

transforms_data = load_json(TRANSFORMS_PATH)
frames = transforms_data.get('frames', [])
if not frames:
    fail(f'No frames in transforms.json: {TRANSFORMS_PATH}')
frame_keys = [
    frame_key_from_any(frame.get('file_path', f'frame_{idx:06d}'))
    for idx, frame in enumerate(frames)
]
missing_images = []
for frame in frames:
    image_path = frame_image_path(frame)
    if image_path is None or not image_path.exists():
        missing_images.append(str(image_path))
if missing_images:
    safe_write_csv(
        REPORT_DIR / 'missing_source_images.csv',
        [{'image_path': path} for path in missing_images],
    )
    fail(
        f'{len(missing_images)} frame images referenced by transforms.json are missing. '
        'See missing_source_images.csv'
    )

SOURCE_TRANSFORMS_HASH = stable_json_hash(transforms_data)
SHARED_CACHE_MANIFEST = prepare_shared_local_image_cache()

accepted_detection_labels = set()
for _, payload in detection_payloads:
    for item in iter_detection_items(payload):
        label_norm = detection_label(item)
        if label_norm and detection_record_is_authoritative(item, label_norm):
            accepted_detection_labels.add(label_norm)

explicit_targets = {normalize_label(label) for label in TARGET_LABELS}
invalid_explicit_targets = sorted(explicit_targets - VOLUMETRIC_LABEL_ALLOWLIST)
if invalid_explicit_targets:
    fail(
        'Explicit targets are not approved for compact volumetric containment: '
        + ', '.join(invalid_explicit_targets)
    )
selected_labels = sorted(
    accepted_detection_labels & VOLUMETRIC_LABEL_ALLOWLIST
    & (explicit_targets if explicit_targets else VOLUMETRIC_LABEL_ALLOWLIST)
)
DISCOVERED_VOLUMETRIC_LABELS = selected_labels

discovery_rows = []
for label_norm in sorted(accepted_detection_labels | explicit_targets):
    if label_norm in VOLUMETRIC_LABEL_EXCLUSIONS:
        decision = 'excluded_geometry_policy'
        reason = VOLUMETRIC_LABEL_EXCLUSIONS[label_norm]
    elif label_norm not in VOLUMETRIC_LABEL_ALLOWLIST:
        decision = 'excluded_not_allowlisted'
        reason = 'unknown_geometry_policy'
    elif explicit_targets and label_norm not in explicit_targets:
        decision = 'excluded_by_explicit_filter'
        reason = 'not_requested_for_this_run'
    elif label_norm not in accepted_detection_labels:
        decision = 'unavailable'
        reason = 'no_accepted_masks'
    else:
        decision = 'selected_volumetric_class'
        reason = 'compact_bounded_object_policy'
    discovery_rows.append({
        'label': label_norm,
        'accepted_detection_label': label_norm in accepted_detection_labels,
        'decision': decision,
        'reason': reason,
    })
safe_write_csv(REPORT_DIR / 'volumetric_label_discovery.csv', discovery_rows)

object_audit_rows = []
target_mask_maps = {}
target_mask_source_rows = {}
seen_object_keys = set()
for label_norm in selected_labels:
    _, all_detection_rows = collect_target_masks_from_detections(label_norm)
    candidate_rows = summarize_instance_candidates(all_detection_rows)
    configured_ids = configured_instance_ids(label_norm)
    available_ids = {row['instance_id'] for row in candidate_rows}
    if configured_ids is not None:
        missing_configured = sorted(configured_ids - available_ids)
        if missing_configured:
            fail(
                f'Configured instances for {label_norm!r} were not found: '
                + ', '.join(missing_configured)
            )
    for candidate in candidate_rows:
        instance_id = candidate['instance_id']
        selected_by_filter = configured_ids is None or instance_id in configured_ids
        candidate['selected'] = bool(selected_by_filter)
        candidate['selection_policy'] = INSTANCE_SELECTION_POLICY
        target_key = object_key_for_instance(label_norm, instance_id)
        if target_key in seen_object_keys:
            fail(f'Duplicate normalized target object key: {target_key!r}')
        seen_object_keys.add(target_key)

        if selected_by_filter:
            detection_map, detection_rows = collect_target_masks_from_detections(
                label_norm,
                target_object_id=instance_id,
            )
        else:
            detection_map, detection_rows = {}, []
        if ALLOW_FILENAME_MASK_FALLBACK and selected_by_filter:
            filename_map, filename_rows = collect_target_masks_by_filename(
                label_norm, frame_keys
            )
        else:
            filename_map, filename_rows = {}, []
        frame_to_masks = merge_mask_maps(detection_map, filename_map)
        positive_frames = len(frame_to_masks)
        mask_records = sum(len(values) for values in frame_to_masks.values())
        if not selected_by_filter:
            audit_status = 'excluded_by_instance_filter'
        elif positive_frames < MIN_POSITIVE_MASK_FRAMES:
            audit_status = 'too_few_positive_mask_frames'
        else:
            audit_status = 'ready'
        will_run = (
            selected_by_filter
            and (audit_status == 'ready' or not SKIP_LABELS_WITH_TOO_FEW_MASK_FRAMES)
        )

        if selected_by_filter:
            TARGET_OBJECT_LABELS[target_key] = label_norm
            RESOLVED_INSTANCE_IDS[target_key] = instance_id
            target_mask_maps[target_key] = frame_to_masks
            target_mask_source_rows[target_key] = detection_rows + filename_rows
            TARGET_OBJECTS.append({
                'object_key': target_key,
                'label': label_norm,
                'instance_id': instance_id,
                'positive_mask_frames': positive_frames,
                'will_run': bool(will_run),
            })

        object_audit_rows.append({
            'object_key': target_key,
            'label': label_norm,
            'instance_id': instance_id,
            'audit_status': audit_status,
            'positive_mask_frames': positive_frames,
            'mask_records': mask_records,
            'detection_json_records': len(detection_rows),
            'filename_scan_records': len(filename_rows),
            'will_run': bool(will_run),
        })
        safe_write_csv(
            REPORT_DIR / target_key / 'target_mask_sources.csv',
            target_mask_source_rows.get(target_key, []),
        )
    safe_write_csv(REPORT_DIR / label_norm / 'instance_candidates.csv', candidate_rows)

safe_write_csv(REPORT_DIR / 'batch_object_audit.csv', object_audit_rows)
print('Tracked segmentation fingerprint:', SEGMENTATION_FINGERPRINT)
print('Source frames:', len(frames))
print('Missing images:', len(missing_images))
print('Accepted detection labels:', sorted(accepted_detection_labels))
print('Selected volumetric labels:', DISCOVERED_VOLUMETRIC_LABELS)
print('Target object count:', len(TARGET_OBJECTS))
print('Object audit:')
print_rows(
    object_audit_rows,
    columns=[
        'object_key', 'label', 'instance_id', 'audit_status',
        'positive_mask_frames', 'mask_records', 'will_run',
    ],
    limit=200,
)
print('Label discovery CSV:', REPORT_DIR / 'volumetric_label_discovery.csv')
print('Object audit CSV:', REPORT_DIR / 'batch_object_audit.csv')


## 6. Sparse Point Helpers

In [ ]:
def resolve_ply_file_path_from_transforms(transforms_obj):
    value = transforms_obj.get('ply_file_path')
    candidates = []
    if value:
        p = Path(value)
        if p.is_absolute():
            candidates.append(p)
        else:
            candidates.extend([SOURCE_NS_DATA_DIR / p, SCAN_DIR / p])
    candidates.extend([
        SCAN_DIR / 'sparse_pc.ply',
        SOURCE_NS_DATA_DIR / 'sparse_pc.ply',
        SOURCE_NS_DATA_DIR / 'colmap' / 'sparse_pc.ply',
    ])
    for path in candidates:
        if path.exists():
            return path
    return None


def ply_xyz(vertex):
    names = set(vertex.dtype.names or [])
    if not {'x', 'y', 'z'}.issubset(names):
        fail('Sparse PLY is missing x/y/z fields.')
    return np.column_stack([
        np.asarray(vertex['x'], dtype=np.float32),
        np.asarray(vertex['y'], dtype=np.float32),
        np.asarray(vertex['z'], dtype=np.float32),
    ])


def project_points(points_chunk, frame, transforms_obj, image_shape):
    h, w = image_shape[:2]
    fx, fy, cx, cy = frame_intrinsics(frame, transforms_obj, image_shape)
    c2w = np.asarray(frame['transform_matrix'], dtype=np.float64)
    if c2w.shape != (4, 4):
        fail('Frame transform_matrix is not 4x4.')
    w2c = np.linalg.inv(c2w)
    homo = np.ones((len(points_chunk), 4), dtype=np.float64)
    homo[:, :3] = points_chunk.astype(np.float64)
    cam = (w2c @ homo.T).T[:, :3]
    if CAMERA_CONVENTION == 'nerfstudio_opengl':
        depth_z = -cam[:, 2]
        image_x = cam[:, 0]
        image_y = -cam[:, 1]
    elif CAMERA_CONVENTION == 'opencv':
        depth_z = cam[:, 2]
        image_x = cam[:, 0]
        image_y = cam[:, 1]
    else:
        fail(f'Unknown CAMERA_CONVENTION: {CAMERA_CONVENTION}')
    valid = np.isfinite(depth_z) & (depth_z > 0.05) & (depth_z < 50.0)
    u_float = np.full_like(depth_z, np.nan, dtype=np.float64)
    v_float = np.full_like(depth_z, np.nan, dtype=np.float64)
    u_float[valid] = fx * (image_x[valid] / depth_z[valid]) + cx
    v_float[valid] = fy * (image_y[valid] / depth_z[valid]) + cy
    valid &= np.isfinite(u_float) & np.isfinite(v_float)
    valid &= (u_float >= 0) & (u_float < w) & (v_float >= 0) & (v_float < h)
    idx = np.flatnonzero(valid)
    return (
        idx,
        np.floor(u_float[idx]).astype(np.int32),
        np.floor(v_float[idx]).astype(np.int32),
    )


def write_filtered_vertex_ply(source_ply, output_ply, keep_mask):
    ply = PlyData.read(str(source_ply))
    vertex = ply['vertex'].data
    filtered_vertex = vertex[keep_mask]
    elements = []
    for element in ply.elements:
        if element.name == 'vertex':
            elements.append(PlyElement.describe(filtered_vertex, 'vertex'))
        else:
            elements.append(element)
    PlyData(elements, text=False).write(str(output_ply))


source_sparse_path = resolve_ply_file_path_from_transforms(transforms_data) if CREATE_OBJECT_SPARSE_INIT else None
source_sparse_ply = None
source_sparse_points = None
if CREATE_OBJECT_SPARSE_INIT and source_sparse_path is not None:
    print('Source sparse PLY:', source_sparse_path)
    source_sparse_ply = PlyData.read(str(source_sparse_path))
    source_sparse_points = ply_xyz(source_sparse_ply['vertex'].data)
    print('Source sparse points:', len(source_sparse_points))
elif CREATE_OBJECT_SPARSE_INIT:
    print('No sparse point cloud found. Object sparse initialization will be skipped.')
else:
    print('Object sparse initialization disabled.')

## 7. Batch Pipeline Functions

In [ ]:
def dataset_fingerprint_for_label(label_norm, paths, frame_to_masks):
    return stable_json_hash({
        'pipeline_version': BATCH_PIPELINE_VERSION,
        'source_transforms_hash': SOURCE_TRANSFORMS_HASH,
        'segmentation_fingerprint': SEGMENTATION_FINGERPRINT,
        'object_key': paths['object_key'],
        'label': label_norm,
        'instance_id': paths['instance_id'],
        'mask_inventory': file_inventory_hash(
            [path for masks in frame_to_masks.values() for path in masks]
        ),
        'mask_dilation': mask_dilation_for_label(label_norm),
        'mask_erosion': mask_erosion_for_label(label_norm),
        'downscale_factors': DOWNSCALE_FACTORS_TO_WRITE,
        'coordinate_policy': SHARED_COORDINATE_POLICY,
    })


def build_masked_dataset_for_label(label_norm, paths, frame_to_masks):
    for directory in [paths['root'], paths['ns_data'], paths['mask_dir'], paths['preview_dir'], paths['training_dir'], paths['export_dir'], paths['report_dir']]:
        directory.mkdir(parents=True, exist_ok=True)
    link_shared_images_into_label_dataset(paths)
    dataset_fingerprint = dataset_fingerprint_for_label(label_norm, paths, frame_to_masks)
    if REUSE_COMPLETED_DATASET_STAGE and paths['dataset_stage_manifest'].exists():
        previous = load_json(paths['dataset_stage_manifest'])
        local_transforms = paths['ns_data'] / 'transforms.json'
        if previous.get('fingerprint') == dataset_fingerprint and local_transforms.exists():
            required_masks = suffixed_dataset_dir(paths['mask_dir'], DOWNSCALE_FACTOR)
            if required_masks.exists() and any(required_masks.glob('*.png')):
                print('Reusing completed dataset stage for', label_norm)
                return previous['summary']

    new_transforms = copy.deepcopy(transforms_data)
    new_frames = []
    mask_stats = []
    processed_count = 0
    positive_mask_frame_count = 0
    blank_mask_frame_count = 0
    small_or_unreadable_mask_frame_count = 0

    for idx, (frame, frame_key) in enumerate(tqdm(list(zip(frames, frame_keys)), desc=f'Build dataset: {label_norm}')):
        if MAX_FRAMES is not None and processed_count >= MAX_FRAMES:
            break
        masks = frame_to_masks.get(frame_key, [])
        if not masks and not KEEP_ALL_REGISTERED_FRAMES_FOR_SHARED_COORDINATES:
            continue

        image_path = shared_cached_image_path(frame, 1)
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image is None:
            fail(f'Could not read frame image: {image_path}')
        image_shape = image.shape[:2]

        combined_mask = np.zeros(image_shape, dtype=bool)
        readable_masks = 0
        for mask_path in masks:
            try:
                combined_mask |= read_mask(mask_path, image_shape, label_norm=label_norm)
                readable_masks += 1
            except Exception as exc:
                logger.warning('Skipping unreadable mask %s: %s', mask_path, exc)

        area_fraction = float(combined_mask.mean())
        has_trainable_target_mask = readable_masks > 0 and area_fraction >= MIN_MASK_AREA_FRACTION

        if not has_trainable_target_mask:
            if not KEEP_ALL_REGISTERED_FRAMES_FOR_SHARED_COORDINATES or not EMPTY_MASK_FOR_MISSING_TARGET_FRAMES:
                mask_stats.append({
                    'frame_key': frame_key,
                    'status': 'skipped_too_small_or_unreadable',
                    'readable_masks': readable_masks,
                    'area_fraction': area_fraction,
                    'source_masks': ';'.join(str(p) for p in masks),
                })
                small_or_unreadable_mask_frame_count += 1
                continue
            status = 'included_blank_shared_coordinate_frame' if not masks else 'included_blank_too_small_or_unreadable'
            if masks:
                small_or_unreadable_mask_frame_count += 1
            else:
                blank_mask_frame_count += 1
            combined_mask = np.zeros(image_shape, dtype=bool)
        else:
            status = 'included_positive_target_mask'
            positive_mask_frame_count += 1

        out_image_name = Path(frame.get('file_path', image_path.name)).name
        out_mask_name = f'{Path(out_image_name).stem}_{label_norm}_mask.png'
        written_pyramid_files = write_dataset_masks(
            paths=paths,
            mask=combined_mask,
            out_mask_name=out_mask_name,
        )

        if WRITE_MASKED_RGB_PREVIEWS and has_trainable_target_mask and len(list(paths['preview_dir'].glob('*'))) < PREVIEW_FRAME_LIMIT:
            preview = image.copy()
            preview[~combined_mask] = 0
            cv2.imwrite(str(paths['preview_dir'] / out_image_name), preview)

        new_frame = copy.deepcopy(frame)
        new_frame['file_path'] = f'images/{out_image_name}'
        new_frame['mask_path'] = f'masks/{out_mask_name}'
        new_frame['source_frame_key'] = frame_key
        new_frame['target_label'] = label_norm
        new_frame['target_instance_id'] = paths['instance_id']
        new_frame['target_object_key'] = paths['object_key']
        new_frame['has_target_mask'] = bool(has_trainable_target_mask)
        new_frame['shared_coordinate_frame'] = bool(KEEP_ALL_REGISTERED_FRAMES_FOR_SHARED_COORDINATES)
        new_frames.append(new_frame)
        processed_count += 1

        mask_stats.append({
            'frame_key': frame_key,
            'status': status,
            'readable_masks': readable_masks,
            'area_fraction': area_fraction,
            'has_target_mask': bool(has_trainable_target_mask),
            'source_masks': ';'.join(str(p) for p in masks),
            'output_image': str(paths['image_dir'] / out_image_name),
            'output_mask': str(paths['mask_dir'] / out_mask_name),
            'written_downscale_factors': ','.join(str(item['factor']) for item in written_pyramid_files),
            'shared_cached_image': str(image_path),
        })

    if not new_frames:
        raise RuntimeError(f'No frames survived masked dataset creation for {label_norm}.')
    if positive_mask_frame_count == 0:
        raise RuntimeError(f'No positive masks survived for {label_norm}.')

    new_transforms['frames'] = new_frames
    new_transforms['object_splat_experiment'] = {
        'version': BATCH_PIPELINE_VERSION,
        'target_object_key': paths['object_key'],
        'target_label': label_norm,
        'target_instance_id': paths.get('instance_id'),
        'source_transforms': str(TRANSFORMS_PATH),
        'segmentation_fingerprint': SEGMENTATION_FINGERPRINT,
        'created_at': datetime.now().isoformat(timespec='seconds'),
        'batch_id': BATCH_ID,
        'shared_coordinate_policy': SHARED_COORDINATE_POLICY,
        'keep_all_registered_frames_for_shared_coordinates': KEEP_ALL_REGISTERED_FRAMES_FOR_SHARED_COORDINATES,
        'empty_mask_for_missing_target_frames': EMPTY_MASK_FOR_MISSING_TARGET_FRAMES,
        'original_frame_count': len(frames),
        'object_dataset_frame_count': len(new_frames),
        'positive_mask_frame_count': int(positive_mask_frame_count),
        'blank_mask_frame_count': int(blank_mask_frame_count),
        'small_or_unreadable_mask_frame_count': int(small_or_unreadable_mask_frame_count),
        'mask_dilation_pixels': mask_dilation_for_label(label_norm),
        'mask_erosion_pixels': mask_erosion_for_label(label_norm),
        'min_mask_area_fraction': MIN_MASK_AREA_FRACTION,
        'downscale_factor_for_training': int(DOWNSCALE_FACTOR),
        'downscale_factors_written': [int(value) for value in DOWNSCALE_FACTORS_TO_WRITE],
    }

    alignment_manifest = {
        'schema': 'object_splat_alignment_manifest.v1',
        'version': BATCH_PIPELINE_VERSION,
        'created_at': datetime.now().isoformat(timespec='seconds'),
        'batch_id': BATCH_ID,
        'target_object_key': paths['object_key'],
        'target_label': label_norm,
        'target_instance_id': paths.get('instance_id'),
        'run_id': paths['run_id'],
        'coordinate_policy': SHARED_COORDINATE_POLICY,
        'source_transforms': str(TRANSFORMS_PATH),
        'object_transforms': str(paths['ns_data'] / 'transforms.json'),
        'original_frame_count': len(frames),
        'object_dataset_frame_count': len(new_frames),
        'positive_mask_frame_count': int(positive_mask_frame_count),
        'blank_mask_frame_count': int(blank_mask_frame_count),
        'kept_all_registered_frames': bool(len(new_frames) == len(frames) and MAX_FRAMES is None),
        'expected_scene_transform_to_full_room_splat': {
            'translation': [0.0, 0.0, 0.0],
            'rotation_quaternion_xyzw': [0.0, 0.0, 0.0, 1.0],
            'scale': 1.0,
        },
    }

    safe_write_csv(paths['report_dir'] / 'masked_dataset_frame_report.csv', mask_stats)
    write_json(paths['ns_data'] / 'transforms.json', new_transforms)
    if VERIFY_COORDINATE_INVARIANTS:
        if stable_json_hash({k: v for k, v in new_transforms.items() if k not in {'frames', 'object_splat_experiment', 'ply_file_path'}}) != stable_json_hash({k: v for k, v in transforms_data.items() if k not in {'frames', 'ply_file_path'}}):
            raise RuntimeError(f'Top-level room transform metadata changed for {label_norm}.')
        source_by_key = {frame_key_from_any(f.get('file_path')): f for f in frames}
        for object_frame in new_frames:
            source_frame = source_by_key[object_frame['source_frame_key']]
            if not np.allclose(np.asarray(object_frame['transform_matrix']), np.asarray(source_frame['transform_matrix']), atol=1e-9):
                raise RuntimeError(f'Camera transform changed for {label_norm}: {object_frame["source_frame_key"]}')
    write_json(paths['report_dir'] / 'transforms.json', new_transforms)
    write_json(paths['alignment_manifest'], alignment_manifest)
    write_json(paths['report_dir'] / 'object_splat_alignment_manifest.json', alignment_manifest)

    required_image_dir = suffixed_dataset_dir(paths['image_dir'], DOWNSCALE_FACTOR)
    required_mask_dir = suffixed_dataset_dir(paths['mask_dir'], DOWNSCALE_FACTOR)
    if not required_image_dir.exists() or not any(required_image_dir.glob('*')):
        raise RuntimeError(f'Missing required training image folder/files for {label_norm}: {required_image_dir}')
    if not required_mask_dir.exists() or not any(required_mask_dir.glob('*.png')):
        raise RuntimeError(f'Missing required training mask folder/files for {label_norm}: {required_mask_dir}')

    summary = {
        'object_key': paths['object_key'],
        'label': label_norm,
        'instance_id': paths.get('instance_id'),
        'object_dataset_frame_count': int(len(new_frames)),
        'positive_mask_frame_count': int(positive_mask_frame_count),
        'blank_mask_frame_count': int(blank_mask_frame_count),
        'small_or_unreadable_mask_frame_count': int(small_or_unreadable_mask_frame_count),
        'mean_positive_mask_area': describe_numeric([row['area_fraction'] for row in mask_stats if row.get('status') == 'included_positive_target_mask']).get('mean', 0.0),
        'alignment_manifest': str(paths['alignment_manifest']),
        'shared_image_cache': str(LOCAL_SHARED_DATA_DIR),
        'rgb_images_duplicated_per_label': False,
    }
    write_json(paths['dataset_stage_manifest'], {
        'schema': 'object_splat_dataset_stage.v1',
        'fingerprint': dataset_fingerprint,
        'completed_at': datetime.now().isoformat(timespec='seconds'),
        'summary': summary,
    })
    return summary


def create_sparse_init_for_label(label_norm, paths, mask_stats_path):
    if not CREATE_OBJECT_SPARSE_INIT or source_sparse_path is None or source_sparse_points is None:
        transforms_obj = load_json(paths['ns_data'] / 'transforms.json')
        if REMOVE_FULL_SCENE_PLY_INIT_IF_NO_OBJECT_INIT:
            transforms_obj.pop('ply_file_path', None)
            write_json(paths['ns_data'] / 'transforms.json', transforms_obj)
        return False, 0, str(source_sparse_path) if source_sparse_path else None, None

    rows = safe_read_csv(mask_stats_path)
    positive_rows = [
        row for row in rows
        if row.get('status') == 'included_positive_target_mask'
    ]
    frame_lookup = {frame_key_from_any(f.get('file_path')): f for f in frames}
    hit_counts = np.zeros(len(source_sparse_points), dtype=np.uint16)

    for row in tqdm(
        positive_rows,
        total=len(positive_rows),
        desc=f'Filtering sparse init by masks: {label_norm}',
    ):
        frame = frame_lookup.get(row.get('frame_key'))
        mask_path = Path(row.get('output_mask', ''))
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if frame is None or mask is None:
            continue
        mask = mask > 0
        for start in range(0, len(source_sparse_points), SPARSE_POINT_CHUNK_SIZE):
            end = min(start + SPARSE_POINT_CHUNK_SIZE, len(source_sparse_points))
            local, u, v = project_points(
                source_sparse_points[start:end], frame, transforms_data, mask.shape
            )
            if not len(local):
                continue
            hits = mask[v, u]
            if hits.any():
                hit_counts[start + local[hits]] += 1

    keep = hit_counts >= int(SPARSE_POINT_MIN_MASK_HITS)
    keep_indices = np.flatnonzero(keep)
    if len(keep_indices) > SPARSE_POINT_MAX_OUTPUT_POINTS:
        rank = np.lexsort((keep_indices, -hit_counts[keep_indices]))
        keep_indices = keep_indices[rank[:SPARSE_POINT_MAX_OUTPUT_POINTS]]
        keep = np.zeros(len(keep), dtype=bool)
        keep[keep_indices] = True
    keep_count = int(keep.sum())

    transforms_obj = load_json(paths['ns_data'] / 'transforms.json')
    if keep_count == 0:
        if REMOVE_FULL_SCENE_PLY_INIT_IF_NO_OBJECT_INIT:
            transforms_obj.pop('ply_file_path', None)
            write_json(paths['ns_data'] / 'transforms.json', transforms_obj)
        return False, 0, str(source_sparse_path), None

    object_sparse_path = paths['ns_data'] / f"{paths['object_key']}_sparse_pc.ply"
    write_filtered_vertex_ply(source_sparse_path, object_sparse_path, keep)
    transforms_obj['ply_file_path'] = object_sparse_path.name
    write_json(paths['ns_data'] / 'transforms.json', transforms_obj)
    print('Object sparse PLY written:', object_sparse_path)
    print('Object sparse points:', keep_count, '/', len(source_sparse_points))
    return True, keep_count, str(source_sparse_path), str(object_sparse_path)


def prepare_local_training_stage_for_export(paths):
    persisted_root = paths['training_dir']
    persisted_configs = sorted(persisted_root.glob('**/config.yml'))
    if not persisted_configs:
        raise RuntimeError(f'No config.yml found under persisted training root {persisted_root}')
    persisted_config = persisted_configs[-1]
    persisted_checkpoints = sorted(persisted_config.parent.glob('**/*.ckpt'))
    if not persisted_checkpoints:
        raise RuntimeError(
            f'No Nerfstudio checkpoint .ckpt files found near persisted config {persisted_config}'
        )

    local_root = paths['local_root'] / 'training'
    relative_run_dir = persisted_config.parent.relative_to(persisted_root)
    local_run_dir = local_root / relative_run_dir
    local_config = local_run_dir / persisted_config.name
    local_checkpoints = sorted(local_run_dir.glob('**/*.ckpt')) if local_run_dir.exists() else []
    if local_config.exists() and local_checkpoints:
        return local_config

    if local_run_dir.exists():
        shutil.rmtree(local_run_dir)
    local_run_dir.parent.mkdir(parents=True, exist_ok=True)
    print('Restoring persisted checkpoint to local Colab cache:', local_run_dir)
    subprocess.run(
        [
            'rsync', '-a', '--info=progress2',
            f'{persisted_config.parent}/', f'{local_run_dir}/',
        ],
        check=True,
    )
    local_checkpoints = sorted(local_run_dir.glob('**/*.ckpt'))
    if not local_config.exists() or not local_checkpoints:
        raise RuntimeError(
            f'Checkpoint restore was incomplete. Expected config/checkpoints under {local_run_dir}'
        )
    return local_config


def run_training_for_label(label_norm, paths):
    training_stage_manifest = paths['report_dir'] / 'training_stage_manifest.json'
    paths['report_dir'].mkdir(parents=True, exist_ok=True)
    dataset_stage = load_json(paths['dataset_stage_manifest'])
    method_name = paths.get('method_name', 'splatfacto')
    model_cli_args = [str(value) for value in paths.get('model_cli_args', [])]
    training_fingerprint = stable_json_hash({
        'pipeline_version': BATCH_PIPELINE_VERSION,
        'dataset_fingerprint': dataset_stage['fingerprint'],
        'instance_id': paths.get('instance_id'),
        'variant': paths.get('variant_name'),
        'iterations': train_iterations_for_label(label_norm),
        'downscale_factor': int(DOWNSCALE_FACTOR),
        'model': method_name,
        'model_cli_args': model_cli_args,
        'plugin_sha256': globals().get('PLUGIN_SHA256') if method_name != 'splatfacto' else None,
    })
    if training_stage_manifest.exists():
        previous = load_json(training_stage_manifest)
        config_paths = sorted(paths['training_dir'].glob('**/config.yml'))
        checkpoint_paths = sorted(paths['training_dir'].glob('**/*.ckpt'))
        if previous.get('fingerprint') == training_fingerprint and config_paths and checkpoint_paths:
            print('Reusing completed training stage for', label_norm, paths.get('variant_name'))
            return previous.get('train_log')

    local_data = paths['ns_data']
    local_out = paths['local_root'] / 'training'
    local_log = paths['local_root'] / f"ns_train_{BATCH_ID}_{paths['object_key']}_{paths.get('variant_name', method_name)}.log"
    if local_out.exists():
        shutil.rmtree(local_out)
    local_out.mkdir(parents=True, exist_ok=True)
    if not (local_data / 'transforms.json').exists():
        raise RuntimeError(f'Missing local object dataset transforms: {local_data / "transforms.json"}')

    command = [
        'ns-train', method_name,
        '--max-num-iterations', str(train_iterations_for_label(label_norm)),
        '--steps-per-save', str(int(STEPS_PER_SAVE)),
        '--vis', 'tensorboard',
        '--data', str(local_data),
        '--output-dir', str(local_out),
        *model_cli_args,
        'nerfstudio-data', '--downscale-factor', str(int(DOWNSCALE_FACTOR)),
    ]
    train_cmd = shlex.join(command) + f' 2>&1 | tee {shlex.quote(str(local_log))}'
    print(train_cmd)
    environment = nerfstudio_subprocess_env()
    result = subprocess.run(['bash', '-lc', f'set -o pipefail; {train_cmd}'], env=environment)
    if result.returncode != 0:
        if local_log.exists():
            print('Training log tail:')
            print(''.join(local_log.read_text(errors='replace').splitlines(keepends=True)[-100:]))
        raise RuntimeError(
            f'Training failed for {label_norm}/{paths.get("variant_name")} '
            f'with exit code {result.returncode}. See {local_log}'
        )

    paths['training_dir'].mkdir(parents=True, exist_ok=True)
    subprocess.run(['rsync', '-a', '--info=progress2', f'{local_out}/', f'{paths["training_dir"]}/'], check=True)
    if local_log.exists():
        shutil.copy2(local_log, paths['report_dir'] / local_log.name)
    persisted_log = str(paths['report_dir'] / local_log.name)
    write_json(training_stage_manifest, {
        'schema': 'object_splat_training_stage.v2',
        'fingerprint': training_fingerprint,
        'completed_at': datetime.now().isoformat(timespec='seconds'),
        'instance_id': paths.get('instance_id'),
        'variant': paths.get('variant_name'),
        'method': method_name,
        'iterations': train_iterations_for_label(label_norm),
        'train_log': persisted_log,
    })
    return persisted_log


def export_splat_for_label(label_norm, paths):
    # Nerfstudio serializes the original /content output root into config.yml. A fresh Colab
    # runtime therefore needs the persisted run restored to that same local path before export.
    config_path = prepare_local_training_stage_for_export(paths)
    checkpoint_paths = sorted(config_path.parent.glob('**/*.ckpt'))
    if not checkpoint_paths:
        raise RuntimeError(f'No local Nerfstudio checkpoint .ckpt files found near {config_path}')

    variant_name = paths.get('variant_name', paths.get('method_name', 'splatfacto'))
    local_export = Path(f"/content/object-splat-export-{BATCH_ID}-{paths['object_key']}-{variant_name}")
    export_log = paths['report_dir'] / f'ns_export_{BATCH_ID}_{label_norm}_{variant_name}.log'
    if local_export.exists():
        shutil.rmtree(local_export)
    local_export.mkdir(parents=True, exist_ok=True)

    command = [
        'ns-export', 'gaussian-splat',
        '--load-config', str(config_path),
        '--output-dir', str(local_export),
    ]
    export_cmd = shlex.join(command) + f' 2>&1 | tee {shlex.quote(str(export_log))}'
    print(export_cmd)
    environment = nerfstudio_subprocess_env()
    environment['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
    result = subprocess.run(['bash', '-lc', f'set -o pipefail; {export_cmd}'], env=environment)
    if result.returncode != 0:
        if export_log.exists():
            print('Export log tail:')
            print(''.join(export_log.read_text(errors='replace').splitlines(keepends=True)[-120:]))
        raise RuntimeError(
            f'Export failed for {label_norm}/{variant_name} with exit code {result.returncode}. '
            f'See {export_log}'
        )

    local_files = sorted(str(path) for path in local_export.glob('**/*') if path.is_file())
    if not local_files:
        raise RuntimeError(f'ns-export wrote no files to {local_export}. See {export_log}')
    if paths['export_dir'].exists():
        shutil.rmtree(paths['export_dir'])
    paths['export_dir'].mkdir(parents=True, exist_ok=True)
    subprocess.run(['rsync', '-a', '--info=progress2', f'{local_export}/', f'{paths["export_dir"]}/'], check=True)

    raw_splat_path = paths['export_dir'] / 'splat.ply'
    if not raw_splat_path.exists():
        ply_files = sorted(paths['export_dir'].glob('**/*.ply'))
        if not ply_files:
            raise RuntimeError(f'Export produced no .ply splat under {paths["export_dir"]}')
        shutil.copy2(ply_files[0], raw_splat_path)
    if raw_splat_path.stat().st_size == 0:
        raise RuntimeError(f'Exported splat is empty: {raw_splat_path}')
    drive_files = sorted(str(path) for path in paths['export_dir'].glob('**/*') if path.is_file())
    shutil.rmtree(local_export)
    return {
        'config_path': str(config_path),
        'checkpoint_path': str(checkpoint_paths[-1]),
        'export_log': str(export_log),
        'exported_files': drive_files,
        'raw_splat_path': str(raw_splat_path),
        'splat_path': str(raw_splat_path),
    }


def sigmoid(x):
    x = np.asarray(x, dtype=np.float32)
    return 1.0 / (1.0 + np.exp(-np.clip(x, -60.0, 60.0)))


def vertex_xyz(vertex):
    names = set(vertex.dtype.names or [])
    if not {'x', 'y', 'z'}.issubset(names):
        raise RuntimeError('PLY vertex element is missing x/y/z fields.')
    return np.column_stack([
        np.asarray(vertex['x'], dtype=np.float32),
        np.asarray(vertex['y'], dtype=np.float32),
        np.asarray(vertex['z'], dtype=np.float32),
    ])


def gaussian_alpha_from_vertex(vertex):
    names = set(vertex.dtype.names or [])
    if 'opacity' not in names:
        return None
    opacity = np.asarray(vertex['opacity'], dtype=np.float32)
    if opacity.size == 0:
        return None
    if np.nanmin(opacity) < 0.0 or np.nanmax(opacity) > 1.0:
        return sigmoid(opacity)
    return np.clip(opacity, 0.0, 1.0)


def gaussian_largest_sigma(vertex):
    names = set(vertex.dtype.names or [])
    scale_names = [name for name in ['scale_0', 'scale_1', 'scale_2'] if name in names]
    if not scale_names:
        return np.zeros(len(vertex), dtype=np.float32)
    raw = np.column_stack([np.asarray(vertex[name], dtype=np.float32) for name in scale_names])
    # Nerfstudio Gaussian exports store log-scales. Values already in a plausible positive linear
    # range are still interpreted as logs because that is the format used by splatfacto exports.
    sigma = np.exp(np.clip(raw, -20.0, 10.0))
    return np.max(sigma, axis=1).astype(np.float32)


def write_filtered_ply_like_source(source_ply_path, output_ply_path, keep_mask):
    ply = PlyData.read(str(source_ply_path))
    vertex = ply['vertex'].data
    elements = []
    for element in ply.elements:
        if element.name == 'vertex':
            elements.append(PlyElement.describe(vertex[keep_mask], 'vertex'))
        else:
            elements.append(element)
    PlyData(elements, text=False).write(str(output_ply_path))


def evenly_sample_rows(rows, limit):
    rows = list(rows)
    if limit is None or len(rows) <= int(limit):
        return rows
    indices = np.linspace(0, len(rows) - 1, int(limit), dtype=np.int64)
    return [rows[int(index)] for index in np.unique(indices)]


def project_cleanup_chunk(points, frame, image_shape):
    h, w = image_shape[:2]
    fx, fy, cx, cy = frame_intrinsics(frame, transforms_data, image_shape)
    c2w = np.asarray(frame['transform_matrix'], dtype=np.float64)
    w2c = np.linalg.inv(c2w)
    homo = np.ones((len(points), 4), dtype=np.float64)
    homo[:, :3] = points.astype(np.float64)
    cam = (w2c @ homo.T).T[:, :3]
    if CAMERA_CONVENTION == 'nerfstudio_opengl':
        depth = -cam[:, 2]
        image_x = cam[:, 0]
        image_y = -cam[:, 1]
    else:
        depth = cam[:, 2]
        image_x = cam[:, 0]
        image_y = cam[:, 1]
    valid = np.isfinite(depth) & (depth > 0.05) & (depth < 100.0)
    u = np.full(len(points), np.nan, dtype=np.float64)
    v = np.full(len(points), np.nan, dtype=np.float64)
    u[valid] = fx * image_x[valid] / depth[valid] + cx
    v[valid] = fy * image_y[valid] / depth[valid] + cy
    valid &= np.isfinite(u) & np.isfinite(v)
    valid &= (u >= 0) & (u < w) & (v >= 0) & (v < h)
    index = np.flatnonzero(valid)
    return index, u[index], v[index], depth[index], float(max(fx, fy))


def footprint_mask_coverage(mask, u, v, radius):
    h, w = mask.shape[:2]
    offsets = np.asarray([
        [0.0, 0.0],
        [0.55, 0.0], [-0.55, 0.0], [0.0, 0.55], [0.0, -0.55],
        [0.39, 0.39], [0.39, -0.39], [-0.39, 0.39], [-0.39, -0.39],
    ], dtype=np.float32)
    hits = np.zeros(len(u), dtype=np.float32)
    valid_samples = np.zeros(len(u), dtype=np.float32)
    for ox, oy in offsets:
        sample_u = np.rint(u + ox * radius).astype(np.int64)
        sample_v = np.rint(v + oy * radius).astype(np.int64)
        valid = (sample_u >= 0) & (sample_u < w) & (sample_v >= 0) & (sample_v < h)
        if valid.any():
            hits[valid] += mask[sample_v[valid], sample_u[valid]].astype(np.float32)
            valid_samples[valid] += 1.0
    return hits / np.maximum(valid_samples, 1.0)


def multiview_gaussian_support(xyz, sigma, label_norm, paths):
    frame_lookup = {frame_key_from_any(frame.get('file_path')): frame for frame in frames}
    rows = safe_read_csv(paths['report_dir'] / 'masked_dataset_frame_report.csv')
    positive_rows = evenly_sample_rows(
        [row for row in rows if row.get('status') == 'included_positive_target_mask'],
        CLEANUP_POSITIVE_FRAME_LIMIT,
    )
    negative_rows = evenly_sample_rows(
        [row for row in rows if row.get('status') == 'included_blank_shared_coordinate_frame'],
        CLEANUP_NEGATIVE_FRAME_LIMIT,
    )
    selected = [(row, False) for row in positive_rows] + [(row, True) for row in negative_rows]
    support_views = np.zeros(len(xyz), dtype=np.uint16)
    positive_visible = np.zeros(len(xyz), dtype=np.uint16)
    contradiction_weight = np.zeros(len(xyz), dtype=np.float32)
    total_visible_weight = np.zeros(len(xyz), dtype=np.float32)
    coverage_sum = np.zeros(len(xyz), dtype=np.float32)
    evaluated_frames = []

    for row, is_negative in tqdm(selected, desc=f'Cleanup mask support: {label_norm}'):
        frame = frame_lookup.get(row.get('frame_key'))
        if frame is None:
            continue
        image_path = shared_cached_image_path(frame, 1)
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image is None:
            continue
        h, w = image.shape[:2]
        mask_path = Path(row.get('output_mask', ''))
        if is_negative:
            mask = np.zeros((h, w), dtype=bool)
        else:
            mask_image = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            if mask_image is None:
                continue
            if mask_image.shape[:2] != (h, w):
                mask_image = cv2.resize(mask_image, (w, h), interpolation=cv2.INTER_NEAREST)
            mask = mask_image > 0

        zbuffer = np.full(h * w, np.inf, dtype=np.float32)
        for start_index in range(0, len(xyz), CLEANUP_PROJECTION_CHUNK_SIZE):
            end_index = min(start_index + CLEANUP_PROJECTION_CHUNK_SIZE, len(xyz))
            local, u, v, depth, _ = project_cleanup_chunk(xyz[start_index:end_index], frame, image.shape[:2])
            if len(local) == 0:
                continue
            pixel = np.rint(v).astype(np.int64) * w + np.rint(u).astype(np.int64)
            pixel = np.clip(pixel, 0, h * w - 1)
            np.minimum.at(zbuffer, pixel, depth.astype(np.float32))

        frame_visible = 0
        for start_index in range(0, len(xyz), CLEANUP_PROJECTION_CHUNK_SIZE):
            end_index = min(start_index + CLEANUP_PROJECTION_CHUNK_SIZE, len(xyz))
            local, u, v, depth, focal = project_cleanup_chunk(xyz[start_index:end_index], frame, image.shape[:2])
            if len(local) == 0:
                continue
            global_index = start_index + local
            center_u = np.clip(np.rint(u).astype(np.int64), 0, w - 1)
            center_v = np.clip(np.rint(v).astype(np.int64), 0, h - 1)
            nearest_depth = zbuffer[center_v * w + center_u]
            sigma_local = sigma[global_index]
            visible = depth <= (
                nearest_depth
                + float(CLEANUP_ZBUFFER_ABS_TOLERANCE)
                + float(CLEANUP_ZBUFFER_SIGMA_MULTIPLIER) * sigma_local
            )
            if not visible.any():
                continue
            global_index = global_index[visible]
            u = u[visible]
            v = v[visible]
            depth = depth[visible]
            sigma_local = sigma_local[visible]
            radius = np.clip(
                3.0 * focal * sigma_local / np.maximum(depth, 1e-5),
                1.0,
                float(CLEANUP_MAX_FOOTPRINT_RADIUS_PX),
            )
            coverage = footprint_mask_coverage(mask, u, v, radius)
            view_weight = float(CLEANUP_NEGATIVE_VIEW_WEIGHT if is_negative else 1.0)
            total_visible_weight[global_index] += view_weight
            if is_negative:
                contradiction_weight[global_index] += view_weight
            else:
                positive_visible[global_index] += 1
                coverage_sum[global_index] += coverage
                supported = coverage >= float(CLEANUP_MIN_FOOTPRINT_COVERAGE)
                support_views[global_index[supported]] += 1
                contradicted = coverage <= 0.02
                contradiction_weight[global_index[contradicted]] += 1.0
            frame_visible += int(len(global_index))
        evaluated_frames.append({
            'frame_key': row.get('frame_key'),
            'negative': bool(is_negative),
            'visible_gaussians': frame_visible,
            'mask_path': str(mask_path) if not is_negative else None,
        })

    support_ratio = support_views.astype(np.float32) / np.maximum(positive_visible, 1)
    contradiction_ratio = contradiction_weight / np.maximum(total_visible_weight, 1e-6)
    mean_coverage = coverage_sum / np.maximum(positive_visible, 1)
    strong = (
        (support_views >= int(CLEANUP_MIN_SUPPORTING_VIEWS))
        & (support_ratio >= float(CLEANUP_MIN_MASK_SUPPORT_RATIO))
        & (contradiction_ratio <= float(CLEANUP_MAX_CONTRADICTION_RATIO))
    )
    weak = (support_views >= 1) & (mean_coverage >= 0.10)
    strong_background = (
        (positive_visible >= int(CLEANUP_BACKGROUND_MIN_VISIBLE_VIEWS))
        & (support_views <= int(CLEANUP_BACKGROUND_MAX_SUPPORTING_VIEWS))
        & (contradiction_ratio >= float(CLEANUP_BACKGROUND_MIN_CONTRADICTION_RATIO))
    )
    return {
        'strong': strong,
        'weak': weak,
        'strong_background': strong_background,
        'support_views': support_views,
        'positive_visible': positive_visible,
        'support_ratio': support_ratio,
        'contradiction_ratio': contradiction_ratio,
        'mean_coverage': mean_coverage,
        'evaluated_frames': evaluated_frames,
        'positive_frame_count': len(positive_rows),
        'negative_frame_count': len(negative_rows),
    }


def estimate_sparse_spacing_numpy(sparse_xyz, sample_limit=5000, chunk_size=256):
    if len(sparse_xyz) < 2:
        return 0.0
    sample_count = min(int(sample_limit), len(sparse_xyz))
    if sample_count == len(sparse_xyz):
        sample_index = np.arange(len(sparse_xyz), dtype=np.int64)
    else:
        # Uniform deterministic sampling avoids NumPy's optional random binary while
        # preserving coverage across the input ordering.
        sample_index = np.linspace(
            0,
            len(sparse_xyz) - 1,
            num=sample_count,
            dtype=np.int64,
        )
        sample_index = np.unique(sample_index)
        sample_count = len(sample_index)
    sample = np.asarray(sparse_xyz[sample_index], dtype=np.float32)
    nearest = np.full(sample_count, np.inf, dtype=np.float32)
    for start_index in range(0, sample_count, int(chunk_size)):
        end_index = min(start_index + int(chunk_size), sample_count)
        delta = sample[start_index:end_index, None, :] - sample[None, :, :]
        distance_sq = np.einsum('ijk,ijk->ij', delta, delta)
        row_index = np.arange(end_index - start_index)
        distance_sq[row_index, np.arange(start_index, end_index)] = np.inf
        nearest[start_index:end_index] = np.sqrt(np.min(distance_sq, axis=1))
    valid = nearest[np.isfinite(nearest) & (nearest > 0)]
    if not valid.size:
        return 0.0
    # A sample has wider spacing than the complete cloud. The square-root correction is
    # appropriate for the mostly surface-like sparse geometry in a room.
    sampling_correction = math.sqrt(sample_count / max(len(sparse_xyz), 1))
    return float(np.median(valid) * sampling_correction)


def voxel_neighbor_membership(query_xyz, sparse_xyz, voxel_size):
    query_xyz = np.asarray(query_xyz, dtype=np.float64)
    sparse_xyz = np.asarray(sparse_xyz, dtype=np.float64)
    if len(query_xyz) == 0:
        return np.zeros(0, dtype=bool)
    if len(sparse_xyz) == 0 or not np.isfinite(voxel_size) or voxel_size <= 0:
        return np.zeros(len(query_xyz), dtype=bool)
    all_min = np.minimum(query_xyz.min(axis=0), sparse_xyz.min(axis=0)) - float(voxel_size)
    sparse_voxel = np.floor((sparse_xyz - all_min) / float(voxel_size)).astype(np.int64)
    query_voxel = np.floor((query_xyz - all_min) / float(voxel_size)).astype(np.int64)
    def encode(voxel):
        contiguous = np.ascontiguousarray(voxel, dtype=np.int64)
        return contiguous.view(np.dtype((np.void, contiguous.dtype.itemsize * 3))).reshape(-1)

    occupied = np.unique(encode(sparse_voxel))
    supported = np.zeros(len(query_voxel), dtype=bool)
    for dx in (-1, 0, 1):
        for dy in (-1, 0, 1):
            for dz in (-1, 0, 1):
                shifted = query_voxel + np.asarray([dx, dy, dz], dtype=np.int64)
                valid = np.all(shifted >= 0, axis=1)
                if valid.any():
                    supported[valid] |= np.isin(encode(shifted[valid]), occupied, assume_unique=False)
    return supported


def robust_sparse_geometry_support(xyz, sigma, object_sparse_path):
    if not object_sparse_path or not Path(object_sparse_path).exists():
        return {
            'available': False,
            'keep': np.zeros(len(xyz), dtype=bool),
            'spacing': None,
            'robust_sparse_xyz': None,
            'nearest_distance': None,
            'method': 'numpy_multiresolution_voxel',
        }
    sparse_ply = PlyData.read(str(object_sparse_path))
    sparse_xyz = vertex_xyz(sparse_ply['vertex'].data)
    if len(sparse_xyz) < 8:
        return {
            'available': False,
            'keep': np.zeros(len(xyz), dtype=bool),
            'spacing': None,
            'robust_sparse_xyz': sparse_xyz,
            'nearest_distance': None,
            'method': 'numpy_multiresolution_voxel',
        }

    spacing = estimate_sparse_spacing_numpy(sparse_xyz)
    positive_extent = sigma[np.isfinite(sigma) & (sigma > 0)]
    fallback_scale = float(np.median(positive_extent)) if positive_extent.size else 1e-3
    spacing = max(float(spacing), fallback_scale * 0.5, 1e-5)
    allowed_distance = (
        float(CLEANUP_SPARSE_DISTANCE_SPACING_MULTIPLIER) * spacing
        + float(CLEANUP_GAUSSIAN_EXTENT_MULTIPLIER) * sigma
    )

    # Use several voxel scales so each Gaussian is evaluated at a resolution appropriate to its
    # allowed distance. Checking adjacent voxels protects points near voxel boundaries.
    min_voxel = max(spacing * 0.5, 1e-5)
    max_target = max(float(np.percentile(allowed_distance, 99)) * 0.5, min_voxel)
    voxel_sizes = [min_voxel]
    while voxel_sizes[-1] < max_target and len(voxel_sizes) < 12:
        voxel_sizes.append(voxel_sizes[-1] * 2.0)
    voxel_sizes = np.asarray(voxel_sizes, dtype=np.float64)
    level = np.searchsorted(2.0 * voxel_sizes, allowed_distance, side='left')
    level = np.clip(level, 0, len(voxel_sizes) - 1)

    keep = np.zeros(len(xyz), dtype=bool)
    for level_index, voxel_size in enumerate(voxel_sizes):
        index = np.flatnonzero(level == level_index)
        if len(index) == 0:
            continue
        for start_index in range(0, len(index), CLEANUP_PROJECTION_CHUNK_SIZE):
            selected = index[start_index:start_index + CLEANUP_PROJECTION_CHUNK_SIZE]
            keep[selected] = voxel_neighbor_membership(xyz[selected], sparse_xyz, float(voxel_size))

    return {
        'available': True,
        'keep': keep,
        'spacing': spacing,
        'robust_sparse_xyz': sparse_xyz,
        'nearest_distance': None,
        'method': 'numpy_multiresolution_voxel',
        'voxel_sizes': voxel_sizes.astype(float).tolist(),
    }

def robust_surface_plane_keep(xyz, sigma, sparse_xyz, spacing):
    if sparse_xyz is None or len(sparse_xyz) < 20:
        return np.ones(len(xyz), dtype=bool), None
    threshold = max(float(CLEANUP_PLANE_SPACING_MULTIPLIER) * max(float(spacing or 0.0), 1e-6), 1e-4)
    best_inliers = None
    best_plane = None
    state = 42
    point_count = len(sparse_xyz)
    for _ in range(int(CLEANUP_PLANE_RANSAC_ITERATIONS)):
        # A small deterministic LCG supplies three distinct indices without importing
        # numpy.random. Reproducibility is useful for cleanup diagnostics as well.
        sample_index = []
        attempts = 0
        while len(sample_index) < 3 and attempts < 32:
            state = (1664525 * state + 1013904223) & 0xFFFFFFFF
            candidate = int(state % point_count)
            if candidate not in sample_index:
                sample_index.append(candidate)
            attempts += 1
        if len(sample_index) < 3:
            continue
        sample = sparse_xyz[np.asarray(sample_index, dtype=np.int64)]
        normal = np.cross(sample[1] - sample[0], sample[2] - sample[0])
        norm = float(np.linalg.norm(normal))
        if norm < 1e-8:
            continue
        normal = normal / norm
        offset = -float(np.dot(normal, sample[0]))
        residual = np.abs(sparse_xyz @ normal + offset)
        inliers = residual <= threshold
        if best_inliers is None or int(inliers.sum()) > int(best_inliers.sum()):
            best_inliers = inliers
            best_plane = (normal, offset)
    if best_inliers is None or int(best_inliers.sum()) < 10:
        return np.ones(len(xyz), dtype=bool), None
    inlier_xyz = sparse_xyz[best_inliers]
    center = np.mean(inlier_xyz, axis=0)
    _, _, vh = np.linalg.svd(inlier_xyz - center, full_matrices=False)
    normal = vh[-1]
    normal = normal / max(float(np.linalg.norm(normal)), 1e-8)
    offset = -float(np.dot(normal, center))
    distance = np.abs(xyz @ normal + offset)
    allowed = threshold + float(CLEANUP_GAUSSIAN_EXTENT_MULTIPLIER) * sigma
    keep = distance <= allowed
    return keep, {
        'normal': normal.astype(float).tolist(),
        'offset': offset,
        'threshold': threshold,
        'sparse_inliers': int(best_inliers.sum()),
        'sparse_total': int(len(sparse_xyz)),
    }


def component_supported_keep(xyz, base_keep, strong_visual, spacing, sigma):
    index = np.flatnonzero(base_keep)
    if len(index) == 0:
        return base_keep.copy(), []
    selected_xyz = xyz[index]
    selected_sigma = sigma[index]
    median_sigma = float(np.median(selected_sigma)) if len(selected_sigma) else 0.0
    voxel_size = max(
        float(CLEANUP_VOXEL_SPACING_MULTIPLIER) * max(float(spacing or 0.0), 1e-6),
        2.0 * median_sigma,
        1e-4,
    )
    voxel = np.floor(selected_xyz / voxel_size).astype(np.int64)
    unique_voxel, inverse, counts = np.unique(voxel, axis=0, return_inverse=True, return_counts=True)
    visual_counts = np.bincount(inverse, weights=strong_visual[index].astype(np.int64), minlength=len(unique_voxel))
    lookup = {tuple(value): idx for idx, value in enumerate(unique_voxel)}
    visited = np.zeros(len(unique_voxel), dtype=bool)
    component_rows = []
    keep_voxel = np.zeros(len(unique_voxel), dtype=bool)
    neighbor_offsets = [
        (dx, dy, dz)
        for dx in (-1, 0, 1)
        for dy in (-1, 0, 1)
        for dz in (-1, 0, 1)
        if (dx, dy, dz) != (0, 0, 0)
    ]
    minimum_count = max(
        int(CLEANUP_COMPONENT_MIN_GAUSSIANS),
        int(math.ceil(len(index) * float(CLEANUP_COMPONENT_MIN_FRACTION))),
    )
    for seed in range(len(unique_voxel)):
        if visited[seed]:
            continue
        stack = [seed]
        visited[seed] = True
        members = []
        while stack:
            current = stack.pop()
            members.append(current)
            vx, vy, vz = unique_voxel[current]
            for dx, dy, dz in neighbor_offsets:
                neighbor = lookup.get((int(vx + dx), int(vy + dy), int(vz + dz)))
                if neighbor is not None and not visited[neighbor]:
                    visited[neighbor] = True
                    stack.append(neighbor)
        gaussian_count = int(counts[members].sum())
        strong_count = int(visual_counts[members].sum())
        accepted = (
            gaussian_count >= minimum_count
            or strong_count >= int(CLEANUP_COMPONENT_MIN_STRONG_VISUAL_GAUSSIANS)
        )
        if accepted:
            keep_voxel[members] = True
        component_rows.append({
            'component_id': len(component_rows),
            'voxel_count': len(members),
            'gaussian_count': gaussian_count,
            'strong_visual_gaussian_count': strong_count,
            'accepted': bool(accepted),
        })
    result = np.zeros(len(xyz), dtype=bool)
    result[index] = keep_voxel[inverse]
    return result, component_rows


def candidate_is_valid(mask, alpha_count):
    count = int(mask.sum())
    completeness_vs_alpha = float(count / max(alpha_count, 1))
    return (
        count >= int(POST_EXPORT_MIN_GAUSSIAN_COUNT)
        and completeness_vs_alpha >= float(CLEANUP_MIN_COMPLETENESS_VS_ALPHA)
    )


_ROOM_SEMANTIC_REFERENCE_CACHE = None


def load_room_semantic_reference():
    global _ROOM_SEMANTIC_REFERENCE_CACHE
    if _ROOM_SEMANTIC_REFERENCE_CACHE is not None:
        return _ROOM_SEMANTIC_REFERENCE_CACHE

    missing = [path for path in [ROOM_LABELED_SPLAT_PATH, ROOM_SEMANTIC_LABELS_PATH] if not path.exists()]
    if missing:
        message = (
            'Room-reference cleanup requires pipeline.ipynb semantic outputs. Missing: '
            + ', '.join(str(path) for path in missing)
        )
        if ROOM_REFERENCE_REQUIRED:
            raise FileNotFoundError(message)
        logger.warning(message)
        return None

    package = load_json(ROOM_SEMANTIC_LABELS_PATH)
    labels = [normalize_label(value) for value in package.get('labels', [])]
    if not labels:
        raise RuntimeError(f'No label list found in {ROOM_SEMANTIC_LABELS_PATH}')

    room_ply = PlyData.read(str(ROOM_LABELED_SPLAT_PATH))
    vertex = room_ply['vertex'].data
    required_fields = {'x', 'y', 'z', 'semantic_label_id', 'semantic_confidence', 'semantic_support_views'}
    missing_fields = sorted(required_fields - set(vertex.dtype.names or ()))
    if missing_fields:
        raise RuntimeError(
            f'{ROOM_LABELED_SPLAT_PATH} is missing semantic fields: {missing_fields}. '
            'Rerun the current pipeline.ipynb against the full unsampled room splat.'
        )

    xyz = vertex_xyz(vertex)
    label_ids = np.asarray(vertex['semantic_label_id'], dtype=np.int32)
    confidence = np.asarray(vertex['semantic_confidence'], dtype=np.float32)
    support_views = np.asarray(vertex['semantic_support_views'], dtype=np.uint16)
    if not (len(xyz) == len(label_ids) == len(confidence) == len(support_views)):
        raise RuntimeError('Room semantic reference arrays have inconsistent lengths.')

    valid_label = (label_ids >= 0) & (label_ids < len(labels))
    reliable = (
        valid_label
        & (confidence >= float(ROOM_REFERENCE_MIN_CONFIDENCE))
        & (support_views >= int(ROOM_REFERENCE_MIN_SUPPORT_VIEWS))
    )
    sigma = gaussian_largest_sigma(vertex)
    spacing = estimate_sparse_spacing_numpy(xyz)
    positive_sigma = sigma[np.isfinite(sigma) & (sigma > 0)]
    median_sigma = float(np.median(positive_sigma)) if positive_sigma.size else 0.0
    sigma_cap = float(np.percentile(positive_sigma, ROOM_REFERENCE_MAX_RADIUS_PERCENTILE)) if positive_sigma.size else 0.0
    radius = max(
        float(ROOM_REFERENCE_MIN_RADIUS),
        float(ROOM_REFERENCE_RADIUS_SPACING_MULTIPLIER) * max(float(spacing), 0.0),
        float(ROOM_REFERENCE_RADIUS_SIGMA_MULTIPLIER) * median_sigma,
    )
    if sigma_cap > 0:
        radius = min(radius, max(float(ROOM_REFERENCE_MIN_RADIUS), 2.0 * sigma_cap))

    _ROOM_SEMANTIC_REFERENCE_CACHE = {
        'xyz': xyz,
        'label_ids': label_ids,
        'confidence': confidence,
        'support_views': support_views,
        'labels': labels,
        'reliable': reliable,
        'spacing': float(spacing),
        'radius': float(radius),
        'reliable_count': int(reliable.sum()),
    }
    print('Loaded room semantic reference:', ROOM_LABELED_SPLAT_PATH)
    print('Room Gaussians:', len(xyz), 'reliable semantic Gaussians:', int(reliable.sum()), 'radius:', radius)
    return _ROOM_SEMANTIC_REFERENCE_CACHE


def voxel_neighbor_count(query_xyz, reference_xyz, voxel_size, chunk_size=250_000):
    query_xyz = np.asarray(query_xyz, dtype=np.float64)
    reference_xyz = np.asarray(reference_xyz, dtype=np.float64)
    result = np.zeros(len(query_xyz), dtype=np.uint32)
    if len(query_xyz) == 0 or len(reference_xyz) == 0 or not np.isfinite(voxel_size) or voxel_size <= 0:
        return result

    origin = np.minimum(query_xyz.min(axis=0), reference_xyz.min(axis=0)) - float(voxel_size)
    reference_voxel = np.floor((reference_xyz - origin) / float(voxel_size)).astype(np.int64)
    query_voxel = np.floor((query_xyz - origin) / float(voxel_size)).astype(np.int64)
    maximum = np.maximum(reference_voxel.max(axis=0), query_voxel.max(axis=0)) + 3
    stride_y = int(maximum[0] + 1)
    stride_z = int((maximum[0] + 1) * (maximum[1] + 1))

    def encode(voxel):
        return voxel[:, 0] + stride_y * voxel[:, 1] + stride_z * voxel[:, 2]

    reference_keys, reference_counts = np.unique(encode(reference_voxel), return_counts=True)
    offsets = np.asarray(
        [(dx, dy, dz) for dx in (-1, 0, 1) for dy in (-1, 0, 1) for dz in (-1, 0, 1)],
        dtype=np.int64,
    )
    for start_index in range(0, len(query_voxel), int(chunk_size)):
        end_index = min(start_index + int(chunk_size), len(query_voxel))
        local_voxel = query_voxel[start_index:end_index]
        local_count = np.zeros(len(local_voxel), dtype=np.uint32)
        for offset in offsets:
            keys = encode(local_voxel + offset)
            position = np.searchsorted(reference_keys, keys)
            valid = position < len(reference_keys)
            matched = np.zeros(len(keys), dtype=bool)
            matched[valid] = reference_keys[position[valid]] == keys[valid]
            local_count[matched] += reference_counts[position[matched]].astype(np.uint32)
        result[start_index:end_index] = local_count
    return result


def room_reference_conflict_keep(label_norm, xyz):
    reference = load_room_semantic_reference()
    if reference is None:
        return np.ones(len(xyz), dtype=bool), {'available': False, 'reason': 'reference_not_available'}

    matching_label_ids = [index for index, value in enumerate(reference['labels']) if value == label_norm]
    if len(matching_label_ids) != 1:
        return np.ones(len(xyz), dtype=bool), {
            'available': False,
            'reason': 'target_label_missing_or_ambiguous_in_room_reference',
            'matching_label_ids': matching_label_ids,
        }
    target_id = matching_label_ids[0]
    reliable = reference['reliable']
    target_reference = reliable & (reference['label_ids'] == target_id)
    competing_reference = reliable & (reference['label_ids'] != target_id)
    if int(target_reference.sum()) < int(ROOM_REFERENCE_MIN_TARGET_POINTS):
        return np.ones(len(xyz), dtype=bool), {
            'available': False,
            'reason': 'too_few_reliable_target_reference_gaussians',
            'target_label_id': int(target_id),
            'target_reference_count': int(target_reference.sum()),
        }

    radius = float(reference['radius'])
    voxel_size = max(radius * 0.5, float(ROOM_REFERENCE_MIN_RADIUS))
    room_count = voxel_neighbor_count(xyz, reference['xyz'], voxel_size)
    target_count = voxel_neighbor_count(xyz, reference['xyz'][target_reference], voxel_size)
    dominant_competing_count = np.zeros(len(xyz), dtype=np.uint32)
    dominant_competing_id = np.full(len(xyz), -1, dtype=np.int32)
    for label_id, label_name in enumerate(reference['labels']):
        if label_id == target_id:
            continue
        label_reference = reliable & (reference['label_ids'] == label_id)
        if not label_reference.any():
            continue
        counts = voxel_neighbor_count(xyz, reference['xyz'][label_reference], voxel_size)
        better = counts > dominant_competing_count
        dominant_competing_count[better] = counts[better]
        dominant_competing_id[better] = int(label_id)
    inside_room = room_count >= int(ROOM_REFERENCE_MIN_ROOM_NEIGHBORS)
    target_protected = target_count >= int(ROOM_REFERENCE_TARGET_PROTECTION_NEIGHBORS)
    competing_conflict = dominant_competing_count >= int(ROOM_REFERENCE_MIN_COMPETING_NEIGHBORS)
    remove = inside_room & competing_conflict & ~target_protected

    removals_by_label = {}
    removed_competing_id = dominant_competing_id[remove]
    for label_id in np.unique(removed_competing_id[removed_competing_id >= 0]):
        removals_by_label[reference['labels'][int(label_id)]] = int((removed_competing_id == label_id).sum())

    report = {
        'available': True,
        'room_labeled_splat': str(ROOM_LABELED_SPLAT_PATH),
        'room_semantic_labels': str(ROOM_SEMANTIC_LABELS_PATH),
        'target_label_id': int(target_id),
        'target_reference_count': int(target_reference.sum()),
        'competing_reference_count': int(competing_reference.sum()),
        'reliable_room_reference_count': int(reliable.sum()),
        'room_reference_radius': radius,
        'voxel_size': voxel_size,
        'inside_room_count': int(inside_room.sum()),
        'target_protected_count': int(target_protected.sum()),
        'competing_conflict_count': int(competing_conflict.sum()),
        'removed_conflict_count': int(remove.sum()),
        'uncertain_retained_count': int((~inside_room | (~target_protected & ~competing_conflict)).sum()),
        'removals_by_competing_label': removals_by_label,
    }
    return ~remove, report


def reuse_previous_raw_export(label_norm, paths):
    if not REUSE_RAW_EXPORTS_FROM_BATCH_ID:
        return None
    source = (
        SCAN_DIR / 'object_splat_batches' / REUSE_RAW_EXPORTS_FROM_BATCH_ID
        / label_norm / 'exports' / 'splat.ply'
    )
    if not source.exists() or source.stat().st_size == 0:
        return None
    paths['export_dir'].mkdir(parents=True, exist_ok=True)
    destination = paths['export_dir'] / 'splat.ply'
    shutil.copy2(source, destination)
    print('Reused previous raw export for cleanup:', source)
    return {
        'raw_splat_path': str(destination),
        'splat_path': str(destination),
        'reused_raw_export_from': str(source),
        'exported_files': [str(destination)],
    }


def cleanup_exported_splat_for_label(label_norm, paths, raw_splat_path, object_sparse_path):
    if not RUN_POST_EXPORT_CLEANUP:
        return {'cleaned_written': False, 'manifest_splat_path': str(raw_splat_path), 'cleanup_reason': 'disabled'}
    raw_splat_path = Path(raw_splat_path)
    if not raw_splat_path.exists():
        raise RuntimeError(f'Cannot clean missing raw splat for {label_norm}: {raw_splat_path}')

    ply = PlyData.read(str(raw_splat_path))
    vertex = ply['vertex'].data
    xyz = vertex_xyz(vertex)
    sigma = gaussian_largest_sigma(vertex)
    positive_sigma = sigma[np.isfinite(sigma) & (sigma > 0)]
    if positive_sigma.size:
        sigma_cap = float(np.percentile(positive_sigma, CLEANUP_MAX_SIGMA_PERCENTILE))
        effective_sigma = np.minimum(sigma, sigma_cap)
    else:
        sigma_cap = 0.0
        effective_sigma = sigma

    alpha = gaussian_alpha_from_vertex(vertex)
    alpha_keep = np.ones(len(vertex), dtype=bool) if alpha is None else alpha >= float(POST_EXPORT_MIN_ALPHA)
    room_keep, room_reference_report = room_reference_conflict_keep(label_norm, xyz)
    room_conflict_keep = alpha_keep & room_keep

    # v5 image-evidence cleanup is retained only as an optional diagnostic. It cannot control the
    # manifest because absence of image evidence previously deleted valid object geometry.
    visual = None
    geometry = None
    plane_report = None
    component_rows = []
    aggressive_evidence_keep = None
    aggressive_final_keep = None
    if RUN_LEGACY_IMAGE_CLEANUP_DIAGNOSTICS:
        visual = multiview_gaussian_support(xyz, effective_sigma, label_norm, paths)
        geometry = robust_sparse_geometry_support(xyz, effective_sigma, object_sparse_path)
        aggressive_evidence_keep = alpha_keep & (
            visual['strong'] | (geometry['keep'] & visual['weak'])
        )
        if label_norm in STRUCTURAL_PLANE_LABELS and geometry['available']:
            plane_keep, plane_report = robust_surface_plane_keep(
                xyz, effective_sigma, geometry['robust_sparse_xyz'], geometry['spacing']
            )
            aggressive_final_keep = aggressive_evidence_keep & plane_keep
        else:
            aggressive_final_keep, component_rows = component_supported_keep(
                xyz,
                aggressive_evidence_keep,
                visual['strong'],
                geometry.get('spacing'),
                effective_sigma,
            )

    # The room-conflict result is selected only if the semantic reference was available and the
    # completeness safeguard passes. Alpha-only is the automatic fallback.
    candidates = [
        ('room_reference_conflict', room_conflict_keep, paths['export_dir'] / 'splat_room_conflict_cleaned.ply', bool(room_reference_report.get('available'))),
        ('alpha_only', alpha_keep, paths['export_dir'] / 'splat_alpha_cleaned.ply', True),
    ]
    if RUN_LEGACY_IMAGE_CLEANUP_DIAGNOSTICS:
        candidates.extend([
            ('aggressive_multiview_geometry', aggressive_evidence_keep, paths['export_dir'] / 'splat_mask_geometry_supported.ply', False),
            ('aggressive_component_or_plane', aggressive_final_keep, paths['export_dir'] / 'splat_aggressive_component_or_plane.ply', False),
        ])
    candidate_rows = []
    selected = None
    alpha_count = int(alpha_keep.sum())
    for name, keep_mask, output_path, selectable in candidates:
        count = int(keep_mask.sum())
        ratio = float(count / max(len(vertex), 1))
        completeness_vs_alpha = float(count / max(alpha_count, 1))
        valid = bool(selectable and candidate_is_valid(keep_mask, alpha_count))
        if count > 0:
            write_filtered_ply_like_source(raw_splat_path, output_path, keep_mask)
        candidate_rows.append({
            'name': name,
            'path': str(output_path) if count > 0 else None,
            'gaussian_count': count,
            'keep_ratio': ratio,
            'completeness_vs_alpha': completeness_vs_alpha,
            'selectable': bool(selectable),
            'valid': bool(valid),
        })
        if selected is None and valid:
            selected = {
                'name': name,
                'path': output_path,
                'keep_mask': keep_mask,
                'count': count,
                'ratio': ratio,
            }

    cleanup_report = {
        'schema': 'object_splat_room_reference_cleanup.v3',
        'version': BATCH_PIPELINE_VERSION,
        'label': label_norm,
        'raw_splat_path': str(raw_splat_path),
        'raw_vertex_count': int(len(vertex)),
        'post_export_min_alpha': float(POST_EXPORT_MIN_ALPHA),
        'alpha_keep_count': int(alpha_keep.sum()),
        'sigma_cap': sigma_cap,
        'room_reference': room_reference_report,
        'room_conflict_keep_count': int(room_conflict_keep.sum()),
        'legacy_image_cleanup_diagnostics_enabled': bool(RUN_LEGACY_IMAGE_CLEANUP_DIAGNOSTICS),
        'visual_positive_frames': visual['positive_frame_count'] if visual else 0,
        'visual_negative_frames': visual['negative_frame_count'] if visual else 0,
        'strong_visual_count': int(visual['strong'].sum()) if visual else 0,
        'weak_visual_count': int(visual['weak'].sum()) if visual else 0,
        'geometry_support_available': bool(geometry['available']) if geometry else False,
        'geometry_support_count': int(geometry['keep'].sum()) if geometry else 0,
        'sparse_spacing': geometry.get('spacing') if geometry else None,
        'aggressive_evidence_keep_count': int(aggressive_evidence_keep.sum()) if aggressive_evidence_keep is not None else 0,
        'plane_report': plane_report,
        'component_count': len(component_rows),
        'components': component_rows,
        'candidates': candidate_rows,
        'evaluated_frames': visual['evaluated_frames'] if visual else [],
    }

    if selected is None:
        cleanup_report.update({
            'cleaned_written': False,
            'cleanup_reason': 'no_selectable_candidate_passed_absolute_count_and_completeness_gate',
            'manifest_splat_path': str(raw_splat_path),
        })
    else:
        canonical_cleaned_path = paths['export_dir'] / 'splat_cleaned.ply'
        if Path(selected['path']) != canonical_cleaned_path:
            shutil.copy2(selected['path'], canonical_cleaned_path)
        cleanup_report.update({
            'cleaned_written': True,
            'selected_candidate': selected['name'],
            'selected_candidate_path': str(selected['path']),
            'cleaned_splat_path': str(canonical_cleaned_path),
            'cleaned_vertex_count': int(selected['count']),
            'cleaned_keep_ratio': float(selected['ratio']),
            'manifest_splat_path': str(canonical_cleaned_path if PREFER_CLEANED_SPLAT_IN_MANIFEST else raw_splat_path),
        })

    safe_write_csv(paths['report_dir'] / 'post_export_cleanup_candidates.csv', candidate_rows)
    safe_write_csv(paths['report_dir'] / 'post_export_cleanup_components.csv', component_rows)
    write_json(paths['report_dir'] / 'post_export_cleanup_report.json', cleanup_report)
    return cleanup_report




def _sample_rows_evenly(rows, limit):
    rows = list(rows)
    if limit is None or len(rows) <= int(limit):
        return rows
    indices = np.linspace(0, len(rows) - 1, num=int(limit), dtype=int)
    return [rows[index] for index in sorted(set(indices.tolist()))]


def _nearest_occupied_indices_6n(occupancy):
    occupied_indices = np.argwhere(occupancy)
    if not len(occupied_indices):
        raise RuntimeError('Cannot build nearest-cell map for an empty support grid.')
    index_dtype = np.int16 if max(occupancy.shape) < np.iinfo(np.int16).max else np.int32
    nearest = np.full(occupancy.shape + (3,), -1, dtype=index_dtype)
    queue = deque()
    for coordinate in occupied_indices:
        item = tuple(int(value) for value in coordinate)
        nearest[item] = coordinate.astype(index_dtype)
        queue.append(item)
    shape = occupancy.shape
    while queue:
        x, y, z = queue.popleft()
        source = nearest[x, y, z]
        for nx, ny, nz in (
            (x - 1, y, z), (x + 1, y, z),
            (x, y - 1, z), (x, y + 1, z),
            (x, y, z - 1), (x, y, z + 1),
        ):
            if nx < 0 or ny < 0 or nz < 0 or nx >= shape[0] or ny >= shape[1] or nz >= shape[2]:
                continue
            if nearest[nx, ny, nz, 0] >= 0:
                continue
            nearest[nx, ny, nz] = source
            queue.append((nx, ny, nz))
    return nearest


def build_visual_support_grid_for_label(label_norm, paths, object_sparse_path):
    object_sparse_path = Path(object_sparse_path) if object_sparse_path else None
    if object_sparse_path is None or not object_sparse_path.exists():
        raise RuntimeError('The containment pilot requires a non-empty object sparse initializer.')
    raw_seed = ply_xyz(PlyData.read(str(object_sparse_path))['vertex'].data).astype(np.float64)
    if len(raw_seed) < 3:
        raise RuntimeError(f'Only {len(raw_seed)} object sparse seeds survived.')

    grid_fingerprint = stable_json_hash({
        'pipeline_version': BATCH_PIPELINE_VERSION,
        'dataset_fingerprint': load_json(paths['dataset_stage_manifest'])['fingerprint'],
        'object_sparse_inventory': file_inventory_hash([object_sparse_path]),
        'instance_id': paths.get('instance_id'),
        'frame_limit': HULL_POSITIVE_FRAME_LIMIT,
        'voxel_spacing_multiplier': HULL_VOXEL_SPACING_MULTIPLIER,
        'padding_fraction': HULL_PADDING_FRACTION,
        'mask_dilation_pixels': HULL_MASK_DILATION_PIXELS,
        'minimum_supporting_views': HULL_MIN_SUPPORTING_VIEWS,
        'minimum_support_ratio': HULL_MIN_SUPPORT_RATIO,
    })
    local_grid_path = paths['ns_data'] / f"{paths['object_key']}_visual_support_grid.npz"
    persisted_grid_path = paths['report_dir'] / f"{paths['object_key']}_visual_support_grid.npz"
    report_path = paths['report_dir'] / 'visual_support_grid_report.json'
    if report_path.exists() and persisted_grid_path.exists():
        previous = load_json(report_path)
        if previous.get('fingerprint') == grid_fingerprint:
            shutil.copy2(persisted_grid_path, local_grid_path)
            print('Reusing visual support grid:', persisted_grid_path)
            return str(local_grid_path), previous

    center = np.median(raw_seed, axis=0)
    _, _, vh = np.linalg.svd(raw_seed - center, full_matrices=False)
    axes = vh.T
    if np.linalg.det(axes) < 0:
        axes[:, -1] *= -1.0
    seed_local = (raw_seed - center) @ axes
    local_min = seed_local.min(axis=0)
    local_max = seed_local.max(axis=0)
    span = np.maximum(local_max - local_min, 1e-6)
    spacing = estimate_sparse_spacing_numpy(raw_seed)
    if not np.isfinite(spacing) or spacing <= 0:
        raise RuntimeError('Could not estimate sparse seed spacing for support grid.')
    padding = np.maximum(
        span * float(HULL_PADDING_FRACTION),
        spacing * float(HULL_PADDING_SPACING_MULTIPLIER),
    )
    local_min -= padding
    local_max += padding
    expanded_span = np.maximum(local_max - local_min, 1e-6)
    voxel_size = max(
        spacing * float(HULL_VOXEL_SPACING_MULTIPLIER),
        float(expanded_span.max()) / HULL_MAX_AXIS_CELLS,
    )
    shape = np.ceil(expanded_span / voxel_size).astype(int) + 1
    total_voxels = int(np.prod(shape))
    if total_voxels > int(HULL_MAX_VOXELS):
        voxel_size *= (total_voxels / float(HULL_MAX_VOXELS)) ** (1.0 / 3.0)
        shape = np.ceil(expanded_span / voxel_size).astype(int) + 1
        total_voxels = int(np.prod(shape))
    if total_voxels > int(HULL_MAX_VOXELS) * 1.10:
        raise RuntimeError(f'Could not bound support-grid size: {shape.tolist()}')

    origin = center + local_min @ axes.T
    flat_index = np.arange(total_voxels, dtype=np.int64)
    grid_index = np.column_stack(
        np.unravel_index(flat_index, tuple(shape))
    ).astype(np.int32)
    candidate_raw = origin + (grid_index.astype(np.float64) * voxel_size) @ axes.T
    projected_count = np.zeros(total_voxels, dtype=np.uint16)
    hit_count = np.zeros(total_voxels, dtype=np.uint16)

    frame_lookup = {frame_key_from_any(frame.get('file_path')): frame for frame in frames}
    positive_rows = [
        row for row in safe_read_csv(paths['report_dir'] / 'masked_dataset_frame_report.csv')
        if row.get('status') == 'included_positive_target_mask'
    ]
    selected_rows = _sample_rows_evenly(positive_rows, HULL_POSITIVE_FRAME_LIMIT)
    for row in tqdm(selected_rows, desc=f'Visual support grid: {label_norm}'):
        frame = frame_lookup.get(row.get('frame_key'))
        mask_path = Path(row.get('output_mask', ''))
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if frame is None or mask is None:
            continue
        mask = mask > 0
        if HULL_MASK_DILATION_PIXELS > 0:
            radius = int(HULL_MASK_DILATION_PIXELS)
            kernel = np.ones((2 * radius + 1, 2 * radius + 1), dtype=np.uint8)
            mask = cv2.dilate(mask.astype(np.uint8), kernel, iterations=1) > 0
        for start in range(0, total_voxels, int(HULL_PROJECTION_CHUNK_SIZE)):
            end = min(start + int(HULL_PROJECTION_CHUNK_SIZE), total_voxels)
            local, u, v = project_points(
                candidate_raw[start:end], frame, transforms_data, mask.shape
            )
            if not len(local):
                continue
            absolute = start + local
            projected_count[absolute] += 1
            hits = mask[v, u]
            hit_count[absolute[hits]] += 1

    support_ratio = hit_count.astype(np.float32) / np.maximum(projected_count, 1)
    occupancy = (
        (hit_count >= int(HULL_MIN_SUPPORTING_VIEWS))
        & (support_ratio >= float(HULL_MIN_SUPPORT_RATIO))
    ).reshape(tuple(shape))

    seed_grid = np.rint(((raw_seed - origin) @ axes) / voxel_size).astype(int)
    seed_grid = np.clip(seed_grid, 0, shape - 1)
    protected_seed_cells = 0
    radius = int(HULL_SEED_PROTECTION_RADIUS_VOXELS)
    for coordinate in seed_grid:
        for dx in range(-radius, radius + 1):
            for dy in range(-radius, radius + 1):
                for dz in range(-radius, radius + 1):
                    cell = coordinate + np.asarray([dx, dy, dz])
                    if np.any(cell < 0) or np.any(cell >= shape):
                        continue
                    cell_tuple = tuple(int(value) for value in cell)
                    if not occupancy[cell_tuple]:
                        protected_seed_cells += 1
                    occupancy[cell_tuple] = True

    occupied_count = int(occupancy.sum())
    if occupied_count < int(HULL_MIN_OCCUPIED_CELLS):
        raise RuntimeError(
            f'Visual support retained only {occupied_count} cells; '
            f'{HULL_MIN_OCCUPIED_CELLS} required.'
        )
    nearest = _nearest_occupied_indices_6n(occupancy)
    np.savez_compressed(
        local_grid_path,
        occupancy=occupancy.astype(np.uint8),
        nearest_occupied_index=nearest,
        origin=origin.astype(np.float32),
        axes=axes.astype(np.float32),
        voxel_size=np.asarray([voxel_size], dtype=np.float32),
    )
    shutil.copy2(local_grid_path, persisted_grid_path)
    report = {
        'schema': 'object_visual_support_grid.v1',
        'fingerprint': grid_fingerprint,
        'created_at': datetime.now().isoformat(timespec='seconds'),
        'object_key': paths['object_key'],
        'label': label_norm,
        'instance_id': paths.get('instance_id'),
        'object_sparse_path': str(object_sparse_path),
        'object_sparse_point_count': int(len(raw_seed)),
        'selected_positive_frames': int(len(selected_rows)),
        'grid_shape': [int(value) for value in shape],
        'total_cells': total_voxels,
        'occupied_cells': occupied_count,
        'occupied_fraction': occupied_count / float(total_voxels),
        'protected_seed_cells_added': int(protected_seed_cells),
        'raw_sparse_spacing': float(spacing),
        'voxel_size': float(voxel_size),
        'persisted_grid_path': str(persisted_grid_path),
    }
    write_json(report_path, report)
    print('Visual support grid:', report)
    return str(local_grid_path), report


def write_batch_manifest(entries, status_rows, batch_complete=True):
    viewer_entries = [entry for entry in entries if entry.get('recommended')]
    successful_objects = sorted({entry['object_key'] for entry in entries})
    viewer_objects = sorted({entry['object_key'] for entry in viewer_entries})
    if successful_objects != viewer_objects:
        raise RuntimeError(
            'Every successful object must have exactly one recommended viewer export. '
            f'Successful={successful_objects}; viewer={viewer_objects}'
        )
    for object_key in successful_objects:
        count = sum(
            1 for entry in viewer_entries if entry['object_key'] == object_key
        )
        if count != 1:
            raise RuntimeError(
                f'Expected one recommended export for {object_key!r}, found {count}.'
            )
    failed = [row for row in status_rows if row.get('status') == 'failed']
    skipped = [row for row in status_rows if row.get('status') == 'skipped']
    manifest = {
        'schema': 'object_splat_manifest.v2',
        'version': BATCH_PIPELINE_VERSION,
        'created_at': datetime.now().isoformat(timespec='seconds'),
        'project_root': str(PROJECT_ROOT),
        'scan_id': SCAN_ID,
        'batch_id': BATCH_ID,
        'source_transforms': str(TRANSFORMS_PATH),
        'coordinate_policy': SHARED_COORDINATE_POLICY,
        'expected_scene_transform_to_full_room_splat': {
            'translation': [0.0, 0.0, 0.0],
            'rotation_quaternion_xyzw': [0.0, 0.0, 0.0, 1.0],
            'scale': 1.0,
        },
        'objects': viewer_entries,
        'status_rows': status_rows,
        'batch_status': (
            'in_progress_checkpoint' if not batch_complete else
            'partial_failure' if failed else
            'success_with_skips' if skipped else
            'success'
        ),
        'volumetric_batch': {
            'auto_discovery_enabled': bool(AUTO_DISCOVER_VOLUMETRIC_LABELS),
            'target_labels': list(DISCOVERED_VOLUMETRIC_LABELS),
            'target_object_keys': [item['object_key'] for item in TARGET_OBJECTS],
            'resolved_instance_ids': dict(RESOLVED_INSTANCE_IDS),
            'object_labels': dict(TARGET_OBJECT_LABELS),
            'instance_selection_policy': INSTANCE_SELECTION_POLICY,
            'recommended_variant': PILOT_RECOMMENDED_VARIANT,
            'variants_run': list(PILOT_VARIANTS_TO_RUN),
            'method_family': 'v9_ellipsoid_containment_all_instances',
            'volumetric_label_allowlist': sorted(VOLUMETRIC_LABEL_ALLOWLIST),
            'excluded_label_policies': dict(VOLUMETRIC_LABEL_EXCLUSIONS),
            'post_export_cleanup_enabled': False,
            'manifest_checkpointed_incrementally': True,
            'batch_complete': bool(batch_complete),
            'completed_object_count': len(successful_objects),
        },
        'ablation_variants': entries,
    }
    write_json(BATCH_MANIFEST_PATH, manifest)
    write_json(LATEST_MANIFEST_PATH, manifest)
    safe_write_csv(REPORT_DIR / 'batch_status.csv', status_rows)
    return manifest



def load_recoverable_variant_entries(eligible_objects):
    recovered_entries = []
    recovered_status_rows = []
    seen_pairs = set()
    for target in eligible_objects:
        object_key = target['object_key']
        label_norm = target['label']
        instance_id = target['instance_id']
        base_paths = label_paths(object_key)
        for variant_name in PILOT_VARIANTS_TO_RUN:
            summary_path = (
                base_paths['report_dir'] / 'variants' / variant_name /
                'variant_summary.json'
            )
            if not summary_path.exists():
                continue
            try:
                entry = load_json(summary_path)
                expected_run_id = f'{object_key}_{variant_name}_{BATCH_ID}'
                if entry.get('status') != 'success':
                    raise RuntimeError('summary status is not success')
                if entry.get('object_key') != object_key:
                    raise RuntimeError('object key does not match the current target')
                if entry.get('label') != label_norm:
                    raise RuntimeError('semantic label does not match the current target')
                if str(entry.get('instance_id')) != str(instance_id):
                    raise RuntimeError('instance ID does not match the current target')
                if entry.get('variant') != variant_name:
                    raise RuntimeError('variant does not match the expected variant')
                if entry.get('run_id') != expected_run_id:
                    raise RuntimeError('run ID does not match this immutable batch')
                splat_path = Path(entry.get('splat_path', ''))
                try:
                    splat_path.relative_to(BATCH_ROOT)
                except ValueError as exc:
                    raise RuntimeError('splat path is outside the current batch root') from exc
                if not splat_path.is_file() or splat_path.stat().st_size <= 0:
                    raise RuntimeError('persisted splat is missing or empty')
            except Exception as exc:
                print('Ignoring invalid recovery summary:', summary_path, exc)
                continue

            pair = (object_key, variant_name)
            if pair in seen_pairs:
                continue
            seen_pairs.add(pair)
            recovered_entries.append(entry)
            recovered_status_rows.append({
                'object_key': object_key,
                'label': label_norm,
                'instance_id': instance_id,
                'variant': variant_name,
                'method': entry.get('method'),
                'status': 'success',
                'stage': 'recovered_complete',
                'positive_mask_frames': entry.get('positive_mask_frame_count'),
                'object_sparse_point_count': entry.get('object_sparse_point_count'),
                'splat_path': str(splat_path),
                'message': f'Recovered from {summary_path}',
            })
    recovered_entries.sort(key=lambda row: (row['object_key'], row['variant']))
    recovered_status_rows.sort(key=lambda row: (row['object_key'], row['variant']))
    return recovered_entries, recovered_status_rows


def checkpoint_batch_progress(entries, status_rows, batch_complete=False):
    safe_write_csv(REPORT_DIR / 'volumetric_variant_status.csv', status_rows)
    if not entries:
        return None
    return write_batch_manifest(
        entries,
        status_rows,
        batch_complete=bool(batch_complete),
    )


def cleanup_local_object_cache(base_paths):
    if not CLEAN_LOCAL_OBJECT_CACHE_AFTER_SUCCESS:
        return
    local_root = Path(base_paths['local_root'])
    if local_root.exists():
        shutil.rmtree(local_root)
        print('Released completed local object cache:', local_root)




## 8. Run Batch Training And Export

In [ ]:
if not VOLUMETRIC_BATCH_MODE:
    fail('This notebook version is intentionally configured as a volumetric-object batch.')
if set(VOLUMETRIC_LABEL_ALLOWLIST) & set(VOLUMETRIC_LABEL_EXCLUSIONS):
    fail('Volumetric allowlist and exclusion policy must be disjoint.')
if PILOT_RECOMMENDED_VARIANT not in PILOT_VARIANTS_TO_RUN:
    fail('The recommended variant must be included in PILOT_VARIANTS_TO_RUN.')

audit_by_object = {
    row['object_key']: row for row in object_audit_rows if row.get('object_key')
}
eligible_objects = [
    item for item in TARGET_OBJECTS
    if item.get('will_run') and audit_by_object[item['object_key']].get('will_run')
]
eligible_objects.sort(
    key=lambda item: (
        item['label'],
        -int(item.get('positive_mask_frames', 0)),
        item['object_key'],
    )
)
if MAX_OBJECTS_TO_RUN is not None:
    eligible_objects = eligible_objects[: int(MAX_OBJECTS_TO_RUN)]
if not eligible_objects:
    fail(
        'No persistent volumetric instance has enough positive mask frames. '
        f'Inspect {REPORT_DIR / "batch_object_audit.csv"}.'
    )

estimated_iterations = sum(
    train_iterations_for_label(item['label']) * len(PILOT_VARIANTS_TO_RUN)
    for item in eligible_objects
)
print('Eligible volumetric objects:', len(eligible_objects))
print('Estimated aggregate training iterations:', estimated_iterations)

status_rows = []
variant_entries = []
eligible_keys = {item['object_key'] for item in eligible_objects}
for row in object_audit_rows:
    if row.get('object_key') not in eligible_keys:
        status_rows.append({
            'object_key': row.get('object_key'),
            'label': row['label'],
            'instance_id': row.get('instance_id'),
            'variant': None,
            'method': None,
            'status': 'skipped',
            'stage': 'audit',
            'positive_mask_frames': row.get('positive_mask_frames', 0),
            'message': row.get('audit_status', 'not_selected'),
        })

recovered_entries, recovered_status_rows = load_recoverable_variant_entries(
    eligible_objects
)
variant_entries.extend(recovered_entries)
status_rows.extend(recovered_status_rows)
completed_pairs = {
    (entry['object_key'], entry['variant']) for entry in variant_entries
}
if recovered_entries:
    checkpoint_batch_progress(variant_entries, status_rows, batch_complete=False)
print('Recovered completed variants:', len(recovered_entries))

for target in eligible_objects:
    target_key = target['object_key']
    label_norm = target['label']
    instance_id = target['instance_id']
    pending_variants = [
        variant_name for variant_name in PILOT_VARIANTS_TO_RUN
        if (target_key, variant_name) not in completed_pairs
    ]
    base_paths = label_paths(target_key)
    if not pending_variants:
        print('\nReusing completed object exports:', target_key)
        cleanup_local_object_cache(base_paths)
        continue

    frame_to_masks = target_mask_maps.get(target_key, {})
    positive_frames = len(frame_to_masks)
    print('\n' + '=' * 80)
    print('Volumetric object:', target_key, label_norm, instance_id)
    print('Positive mask frames:', positive_frames)
    print('Pending variants:', pending_variants)
    base_paths['report_dir'].mkdir(parents=True, exist_ok=True)

    try:
        dataset_summary = build_masked_dataset_for_label(
            label_norm, base_paths, frame_to_masks
        )
        sparse_written, sparse_count, source_sparse, object_sparse_path = (
            create_sparse_init_for_label(
                label_norm,
                base_paths,
                base_paths['report_dir'] / 'masked_dataset_frame_report.csv',
            )
        )
        if not sparse_written or not object_sparse_path:
            raise RuntimeError(
                'V9 ellipsoid containment requires a non-empty object-specific sparse '
                'initializer. Inspect masks and sparse_point_report.csv before changing '
                'SPARSE_POINT_MIN_MASK_HITS.'
            )
        support_grid_path, support_grid_report = build_visual_support_grid_for_label(
            label_norm,
            base_paths,
            object_sparse_path,
        )
    except Exception as exc:
        logger.exception('Volumetric object setup failed: %s', target_key)
        status_rows.append({
            'object_key': target_key,
            'label': label_norm,
            'instance_id': instance_id,
            'variant': None,
            'method': None,
            'status': 'failed',
            'stage': 'dataset_sparse_or_support',
            'positive_mask_frames': positive_frames,
            'message': str(exc),
        })
        checkpoint_batch_progress(variant_entries, status_rows, batch_complete=False)
        if STOP_ON_LABEL_FAILURE:
            raise
        continue

    for variant_name in pending_variants:
        print('\n' + '-' * 80)
        print('Training:', target_key, label_norm, variant_name)
        paths = pilot_variant_paths(
            base_paths,
            variant_name,
            object_sparse_path=object_sparse_path,
            support_grid_path=support_grid_path,
        )
        for directory in [
            paths['root'], paths['training_dir'],
            paths['export_dir'], paths['report_dir'],
        ]:
            directory.mkdir(parents=True, exist_ok=True)
        try:
            train_log = (
                run_training_for_label(label_norm, paths) if RUN_TRAINING else None
            )
            if not RUN_EXPORT:
                raise RuntimeError('RUN_EXPORT must remain enabled for this batch.')
            export_info = export_splat_for_label(label_norm, paths)
            splat_path = Path(export_info['raw_splat_path'])
            entry = {
                'object_id': target_key,
                'object_key': target_key,
                'component_id': f'{target_key}:{variant_name}',
                'label': label_norm,
                'instance_id': instance_id,
                'variant': variant_name,
                'method': paths['method_name'],
                'description': paths['variant_description'],
                'recommended': variant_name == PILOT_RECOMMENDED_VARIANT,
                'run_id': paths['run_id'],
                'splat_path': str(splat_path),
                'raw_splat_path': str(splat_path),
                'splat_path_relative_to_scan': str(splat_path.relative_to(SCAN_DIR)),
                'export_dir': str(paths['export_dir']),
                'training_dir': str(paths['training_dir']),
                'alignment_manifest': str(base_paths['alignment_manifest']),
                'coordinate_policy': SHARED_COORDINATE_POLICY,
                'positive_mask_frame_count': dataset_summary['positive_mask_frame_count'],
                'object_dataset_frame_count': dataset_summary['object_dataset_frame_count'],
                'object_sparse_point_count': int(sparse_count),
                'object_sparse_path': object_sparse_path,
                'visual_support_grid_report': str(
                    base_paths['report_dir'] / 'visual_support_grid_report.json'
                ),
                'post_export_cleanup': False,
                'train_iterations': train_iterations_for_label(label_norm),
                'train_log': train_log,
                'export_log': export_info.get('export_log'),
                'status': 'success',
            }
            write_json(paths['report_dir'] / 'variant_summary.json', entry)
            variant_entries.append(entry)
            completed_pairs.add((target_key, variant_name))
            status_rows.append({
                'object_key': target_key,
                'label': label_norm,
                'instance_id': instance_id,
                'variant': variant_name,
                'method': paths['method_name'],
                'status': 'success',
                'stage': 'complete',
                'positive_mask_frames': dataset_summary['positive_mask_frame_count'],
                'object_sparse_point_count': int(sparse_count),
                'splat_path': str(splat_path),
            })
            checkpoint_batch_progress(variant_entries, status_rows, batch_complete=False)
            print('SUCCESS:', target_key, label_norm, variant_name, splat_path)
        except Exception as exc:
            logger.exception('Variant failed: %s/%s', target_key, variant_name)
            status_rows.append({
                'object_key': target_key,
                'label': label_norm,
                'instance_id': instance_id,
                'variant': variant_name,
                'method': paths['method_name'],
                'status': 'failed',
                'stage': 'training_or_export',
                'positive_mask_frames': positive_frames,
                'message': str(exc),
            })
            checkpoint_batch_progress(variant_entries, status_rows, batch_complete=False)
            if STOP_ON_LABEL_FAILURE:
                raise

    object_complete = all(
        (target_key, variant_name) in completed_pairs
        for variant_name in PILOT_VARIANTS_TO_RUN
    )
    if object_complete:
        cleanup_local_object_cache(base_paths)

if not variant_entries:
    checkpoint_batch_progress(variant_entries, status_rows, batch_complete=False)
    fail('No volumetric-object variant completed successfully. Inspect the status CSV and logs.')

manifest = checkpoint_batch_progress(
    variant_entries,
    status_rows,
    batch_complete=True,
)
print('\nVolumetric-object batch complete:', manifest['batch_status'])
print_rows(
    status_rows,
    columns=[
        'object_key', 'label', 'instance_id', 'variant', 'method',
        'status', 'stage', 'splat_path', 'message',
    ],
    limit=300,
)
print('Viewer-selected variant per successful object:', PILOT_RECOMMENDED_VARIANT)
print('Batch manifest:', BATCH_MANIFEST_PATH)
print('Latest manifest:', LATEST_MANIFEST_PATH)


## 9. Inspect Manifest

In [ ]:
manifest = load_json(LATEST_MANIFEST_PATH)
if manifest.get('version') != BATCH_PIPELINE_VERSION:
    fail(
        f'Manifest version mismatch: {manifest.get("version")!r} '
        f'!= {BATCH_PIPELINE_VERSION!r}'
    )
batch = manifest.get('volumetric_batch', {})
variants = manifest.get('ablation_variants', [])
viewer_objects = manifest.get('objects', [])
if not variants:
    fail('Manifest contains no successful volumetric-object exports.')
successful_objects = sorted({entry['object_key'] for entry in variants})
for object_key in successful_objects:
    selected = [
        entry for entry in viewer_objects
        if entry.get('object_key') == object_key
    ]
    if len(selected) != 1 or selected[0].get('variant') != PILOT_RECOMMENDED_VARIANT:
        fail(
            f'Viewer manifest does not select one recommended export for {object_key!r}.'
        )


def last_containment_log_values(path):
    result = {
        'center_projected_last': None,
        'total_center_projected': None,
        'ellipsoid_shrunk_last': None,
        'total_ellipsoid_shrunk': None,
        'ellipsoid_snapped_last': None,
        'boundary_inside_fraction': None,
    }
    path = Path(path) if path else None
    if path is None or not path.exists():
        return result
    matching = [
        line for line in path.read_text(errors='replace').splitlines()
        if line.startswith('[object-hull] step=')
    ]
    if not matching:
        return result
    values = dict(re.findall(r'([a-z_]+)=([-+0-9.eE]+)', matching[-1]))
    for key in result:
        if key not in values:
            continue
        result[key] = (
            float(values[key])
            if key == 'boundary_inside_fraction'
            else int(float(values[key]))
        )
    return result


inspection_rows = []
for entry in variants:
    splat_path = Path(entry['splat_path'])
    if not splat_path.exists() or splat_path.stat().st_size == 0:
        fail(f'Missing or empty volumetric-object splat: {splat_path}')
    ply = PlyData.read(str(splat_path))
    vertex = ply['vertex'].data
    if len(vertex) <= 0:
        fail(f'Volumetric-object splat has no Gaussians: {splat_path}')
    xyz = vertex_xyz(vertex).astype(np.float64)
    span = np.ptp(xyz, axis=0)
    center = np.mean(xyz, axis=0)
    sigma = gaussian_largest_sigma(vertex).astype(np.float64)
    alpha = gaussian_alpha_from_vertex(vertex)
    alpha = np.asarray(alpha, dtype=np.float64) if alpha is not None else np.asarray([])
    row = {
        'object_key': entry['object_key'],
        'label': entry['label'],
        'instance_id': entry['instance_id'],
        'variant': entry['variant'],
        'method': entry['method'],
        'recommended': entry.get('recommended'),
        'gaussian_count': int(len(vertex)),
        'size_mb': splat_path.stat().st_size / (1024 ** 2),
        'span_x': float(span[0]),
        'span_y': float(span[1]),
        'span_z': float(span[2]),
        'mean_x': float(center[0]),
        'mean_y': float(center[1]),
        'mean_z': float(center[2]),
        'sigma_p50': float(np.percentile(sigma, 50)) if len(sigma) else None,
        'sigma_p90': float(np.percentile(sigma, 90)) if len(sigma) else None,
        'sigma_p99': float(np.percentile(sigma, 99)) if len(sigma) else None,
        'alpha_p50': float(np.percentile(alpha, 50)) if len(alpha) else None,
        'alpha_p90': float(np.percentile(alpha, 90)) if len(alpha) else None,
        'alpha_p99': float(np.percentile(alpha, 99)) if len(alpha) else None,
        'splat_path': str(splat_path),
    }
    row.update(last_containment_log_values(entry.get('train_log')))
    inspection_rows.append(row)

safe_write_csv(REPORT_DIR / 'volumetric_export_inspection.csv', inspection_rows)
write_json(REPORT_DIR / 'volumetric_export_inspection.json', {
    'schema': 'object_splat_volumetric_inspection.v1',
    'pipeline_version': BATCH_PIPELINE_VERSION,
    'batch_id': BATCH_ID,
    'batch_status': manifest.get('batch_status'),
    'target_labels': batch.get('target_labels'),
    'target_object_keys': batch.get('target_object_keys'),
    'resolved_instance_ids': batch.get('resolved_instance_ids'),
    'post_export_cleanup_enabled': batch.get('post_export_cleanup_enabled'),
    'objects': inspection_rows,
})
print('Batch status:', manifest.get('batch_status'))
print('Resolved instances:', batch.get('resolved_instance_ids'))
print('Post-export cleanup enabled:', batch.get('post_export_cleanup_enabled'))
print_rows(
    inspection_rows,
    columns=[
        'object_key', 'label', 'instance_id', 'variant', 'gaussian_count',
        'span_x', 'span_y', 'span_z', 'total_center_projected',
        'total_ellipsoid_shrunk', 'boundary_inside_fraction',
    ],
    limit=100,
)
print('\nViewer-selected object splats:')
for entry in viewer_objects:
    print(entry['object_key'], entry['label'], entry['instance_id'], entry['splat_path'])
print('Inspection CSV:', REPORT_DIR / 'volumetric_export_inspection.csv')
print('Inspection JSON:', REPORT_DIR / 'volumetric_export_inspection.json')
